# FSO-SC: two-stage training

Independent reproduction, not the paper authors' code or released training data.
The semantic model, FP32 loss, Adam and effective batch 8 are unchanged: Stage 1
and Stage 2 each run for 100 epochs. Stage 2 uses the documented controlled BSC
index-error surrogate, not the physical FSO simulator as its training channel.

1. In the setup cell, set `DRIVE_DATA_DIR`, `RUN_NAME` and `START_MODE`.
2. For a fresh experiment use `START_MODE="new"` and an unused run directory.
   The notebook refuses existing files; it never resets or deletes a run.
3. After interruption, use the same data, paths and run name with
   `START_MODE="resume"`. A missing/corrupt full checkpoint does not silently
   become a fresh training run. A completed Stage 1 final can initialize Stage 2.
4. Run cells in order on a Colab GPU. Use only one writer per run.

Provide `imagenet_train_10000.zip`, `GEE_train_fixed.zip` and `GEE_test.zip` in
`DRIVE_DATA_DIR`. Test images are never used in training or best-model selection.
The 10,000+1,000 training images and all preprocessing settings are preserved.
Default fresh initialization seed: 20261029; resuming restores checkpoint RNG.

Full model/Adam/RNG/progress checkpoints are saved every 1,000 updates and at
epoch boundaries. Best means minimum completed-epoch **training** LPIPS, not
validation performance. Latest and best each alternate two bounded slots; only
those run-owned inactive slots and pointers are overwritten. There is no Drive
API transport, trash cleanup, background upload or automatic deletion.

Reserve about 20 GB extra Drive space. Keep the runtime open while a save syncs;
ordinary `drive.mount()` does not guarantee immediate server-side durability.
LPIPS installs from PyPI and may download its pretrained VGG/AlexNet backbones
on first use. No model training is performed by the notebook builder or tests.

Final files are `stage1_paper_protocol_final.pth` and
`stage2_noise_training_final.pth` under `DRIVE_DATA_DIR/checkpoints/RUN_NAME`.
The separate holdout notebook uses the Stage 2 final; optional legacy evaluation
below is disabled by default. Only load checkpoints that you trust: the resume
format contains Python pickle data.


## 명시값과 추정값

| 구분 | 이번 기본값과 근거 |
|---|---|
| 논문 명시 | Fig.2/3 채널 수·블록 수, K16384/d4, GN epsilon1e-6, SN Eq25, MSE+0.1×VGG perceptual+동일 가중치 VQ loss, Adam1e-4/(.5,.9), 학습10000+1000, TableI/II 채널 수치 |
| decoder 순서 | Fig.3 일부 채널/순서 표기가 차원상 모순됨. MoVQ/VQGAN의 일관된 mid Res-Attn-Res, 각 레벨 Res/Attn 배열로 해석. Fig.2 encoder의 표시된 묶음 순서는 유지 |
| 전처리 | 동일 270M 구조의 공개 MoVQ 구현처럼 최대 중앙 정사각 crop → PIL bicubic256/reducing_gap1 → RGB[-1,1]. flip/jitter 없음 |
| batch/안정화 | 공개 MoVQ270M config의 batch8/FP32/value clipping1. 실제 microbatch4×accum2(검사에 따라2×4 또는1×8). 활성값 재계산 해제; 모델 크기 축소 없음 |
| Stage2 미공개 디테일 | q=(0,1e-4,1e-3,5e-3,1e-2)에서 균등 선택. T개 토큰 중 round(T×[1-(1-q)^14])개를 정확히 손상. 각 손상은 Bernoulli(q) 비트 반전의 0이 아닌 XOR mask. q는 실제 측정 BER가 아닌 BSC-reference 값 |
| 학습 예산/옵티마이저 | Stage당 초기100 epochs, Stage2 시작 시 새 Adam. 단계 내부 재개 시 Adam/RNG/데이터 순서를 복구. 별도 학습률 schedule·GAN·EMA·codebook reset 없음 |
| 데이터 원본 | 사용자가 준비한 ImageNet subset 및 NAIP 대체 수집본. 저자 미공개 이미지 목록과 동일하지 않음 |
| 채널 수신기 | symbol-rate matched-filter 근사. heterodyne shot-noise floor와 신호 추정 CFR/CPR을 사용. 블록 진폭 CSI·초기 위상 branch·DSP 파라미터는 추정값. RRC/ADC/timing/IQ 하드웨어 전체는 구현하지 않음 |
| Málaga 식의 미결정성 | 논문이 beta와 coherent phase를 생략. 기존 pi/2 phase/nearest-natural beta 가정을 기록하고, Eq15 moment residual과 불가능 조건을 숨기지 않음. 곡선에 맞춘 파라미터 피팅 없음 |
| 평가 | 원 논문 평가 이미지·반복 수·LPIPS backbone 미공개. 기본 AlexNet LPIPS(공식 패키지 기본값), 3채널 실현 평균, GEE_test 전용. 학습 perceptual loss는 명시된 VGG 유지. 실패 프레임도 통계에 포함 |

근거:

- [MoVQ270M 공개 설정](https://github.com/ai-forever/MoVQGAN/blob/main/configs/movqgan_270M.yaml), [실제 데이터 loader](https://github.com/ai-forever/MoVQGAN/blob/main/movqgan/data/dataset.py). Sber 파생 구현이지 FSO-SC 저자 공개 코드가 아닙니다.
- [저자 연계 Kandinsky MoVQ 모듈](https://github.com/ai-forever/Kandinsky-2/blob/main/kandinsky2/vqgan/movq_modules.py), [VQGAN](https://github.com/CompVis/taming-transformers).
- 원 논문 [31]의 [코드](https://github.com/hqyyqh888/RobustSemanComm/blob/main/channel.py)는 index bits→QAM→AWGN→index 경로를 사용. 여기의 exact-count BSC surrogate/grid는 그 연구에서 가져온 저자 설정이 아니라 본 재현 선택입니다.
- [Pfau et al., blind phase search](https://doi.org/10.1109/JLT.2008.2010511), [Martinez, coherent heterodyne Gaussian noise](https://doi.org/10.1103/PhysRevA.77.032116), [공식 LPIPS](https://github.com/richzhang/PerceptualSimilarity).

절대 LPIPS 곡선 일치를 막는 미공개 사항은 원 데이터, 학습 길이/오류 schedule, optical filter/FOV 단위, 수신 전자잡음, VPI DSP입니다. 이 노트북은 BPG/LDPC 비교군과 Section-V FPGA/10Gbps 실험을 재현하지 않습니다.


## 1. Drive 경로 및 이미지 준비

`내 드라이브/FSO_data/{imagenet_train_10000.zip,GEE_train_fixed.zip,GEE_test.zip}`를 사용합니다. ZIP이 이미 로컬 cache에 있으면 재사용합니다. 기존 파일명은 바꿀 필요 없습니다. GPU 런타임을 선택하세요.


In [ ]:
"""ZIP-to-local-cache adapter. No training, authentication, or downloads at import."""
import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

from PIL import Image


def prepare_uploaded_zip(source_zip, cache_root, dataset_name, expected_count):
    source_zip, cache_root = Path(source_zip), Path(cache_root)
    if dataset_name not in {"imagenet", "gee_train", "gee_test"}:
        raise ValueError("Unknown dataset key")
    if not source_zip.is_file():
        raise FileNotFoundError(f"Drive ZIP not found: {source_zip}")
    archive_dir = cache_root / "archives"
    archive_dir.mkdir(parents=True, exist_ok=True)
    source_stat = source_zip.stat()
    signature = hashlib.sha256(
        f"{source_zip.resolve()}:{source_stat.st_size}:{source_stat.st_mtime_ns}".encode()
    ).hexdigest()[:20]
    local_zip = archive_dir / f"{dataset_name}-{signature}.zip"
    if not local_zip.exists():
        print(f"Copying ZIP to local runtime: {source_zip.name}", flush=True)
        partial = local_zip.with_suffix(".zip.partial")
        shutil.copyfile(source_zip, partial)
        if partial.stat().st_size != source_stat.st_size:
            raise RuntimeError("ZIP copy size differs from source")
        partial.replace(local_zip)
    digest = hashlib.sha256()
    with local_zip.open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024**2), b""):
            digest.update(block)
    zip_sha256 = digest.hexdigest()
    destination = cache_root / "images" / dataset_name / zip_sha256[:20]
    marker = destination / ".prepared.json"
    if marker.exists():
        status = json.loads(marker.read_text(encoding="utf-8"))
        if status != {"zip_sha256": zip_sha256, "image_count": expected_count}:
            raise RuntimeError(f"Unexpected cache marker: {marker}")
        images = [p for p in destination.rglob("*")
                  if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
        if len(images) != expected_count:
            raise RuntimeError(f"Local cache changed: {destination}")
        print(f"Reuse {dataset_name}: {len(images):,} images", flush=True)
        return destination
    staging = destination.with_name(destination.name + ".incomplete")
    staging.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        members = archive.infolist()
        files = []
        for item in members:
            path = PurePosixPath(item.filename)
            if (not path.parts or path.is_absolute() or ".." in path.parts
                    or "\\" in item.filename or ":" in item.filename
                    or stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {item.filename}")
            if not item.is_dir():
                files.append(item.filename)
        if len(files) != len(set(files)):
            raise ValueError("Duplicate ZIP member names")
        image_members = [name for name in files
                         if PurePosixPath(name).suffix.lower() in {".png", ".jpg", ".jpeg"}]
        if len(image_members) != expected_count:
            raise RuntimeError(
                f"{source_zip.name}: expected {expected_count:,} images, "
                f"found {len(image_members):,}"
            )
        remaining = sum(item.file_size for item in members)
        if shutil.disk_usage(staging).free < remaining + 512 * 1024**2:
            raise RuntimeError("Insufficient local runtime disk space for extraction")
        # ZipFile checks CRC while extracting. Only isolated cache contents are written.
        print(f"Extracting {source_zip.name}: {expected_count:,} images", flush=True)
        archive.extractall(staging)
        for name in image_members:
            with Image.open(staging / name) as image:
                image.verify()
    status = {"zip_sha256": zip_sha256, "image_count": expected_count}
    (staging / ".prepared.json").write_text(json.dumps(status), encoding="utf-8")
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite existing folder: {destination}")
    staging.rename(destination)
    print(f"Ready {dataset_name}: {expected_count:,} images", flush=True)
    return destination


def preflight_run_directory(directory, start_mode):
    """Read-only guard: never create, reset, migrate or remove run contents."""
    directory = Path(directory)
    if start_mode not in {"new", "resume"}:
        raise ValueError('START_MODE must be "new" or "resume"')
    if directory.exists() and not directory.is_dir():
        raise NotADirectoryError(directory)
    has_files = directory.exists() and any(path.is_file() for path in directory.rglob("*"))
    if start_mode == "new" and has_files:
        raise FileExistsError("New training requires an empty run directory. "
                              "Choose a different RUN_NAME or explicitly use START_MODE='resume'.")
    if start_mode == "resume" and not has_files:
        raise FileNotFoundError("Resume requires existing run files. "
                                "For a fresh experiment choose START_MODE='new' and an unused RUN_NAME.")


def stage_has_artifacts(directory, stage):
    """Recognize incomplete/orphan stage files as evidence, not permission to reset."""
    return any(path.is_file() and path.name.startswith(f"stage{stage}_")
               for path in Path(directory).rglob("*"))


def seed_fresh_training(seed):
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Configure your own Drive directory and explicit start mode.
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임 유형을 GPU로 변경한 뒤 실행하세요. 학습을 시작하지 않았습니다.")
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/FSO_data")
LOCAL_CACHE = Path("/content/fso_sc_cache")
RUN_NAME = "paper_faithful_public_run1"
START_MODE = "new"  # "new" for an empty run; "resume" for this run only.
FRESH_RUN_SEED = 20261029  # Used only before fresh Stage1 model initialization.
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME은 폴더명 하나로 지정하세요.")
MODEL_OUTPUT_DIR = DRIVE_DATA_DIR / "checkpoints" / RUN_NAME
preflight_run_directory(MODEL_OUTPUT_DIR, START_MODE)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_INPUTS = {
    "imagenet": ("imagenet_train_10000.zip", 10_000),
    "gee_train": ("GEE_train_fixed.zip", 1_000),
    "gee_test": ("GEE_test.zip", 1_000),
}
missing = [str(DRIVE_DATA_DIR / name) for name, _ in ZIP_INPUTS.values()
           if not (DRIVE_DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Drive 파일명/경로를 확인하세요:\n" + "\n".join(missing))
PREPARED_DATA = {
    key: prepare_uploaded_zip(DRIVE_DATA_DIR / name, LOCAL_CACHE, key, count)
    for key, (name, count) in ZIP_INPUTS.items()
}
GEE_TEST_DIR = str(PREPARED_DATA["gee_test"])
assert PREPARED_DATA["gee_train"].resolve() != PREPARED_DATA["gee_test"].resolve()
print("평가 전용 경로 (학습에 포함하지 않음):", GEE_TEST_DIR)
print("최종 가중치 저장 폴더:", MODEL_OUTPUT_DIR)



In [ ]:
import os
import numpy as np
from pathlib import Path
import torch
from PIL import Image
from torch.utils.data import ConcatDataset, Dataset

PAPER_TRAIN_COUNTS = {"imagenet": 10_000, "google_earth": 1_000}
PAPER_DATA_PROTOCOL = "paper_all_11000_training_no_disclosed_split_v1"
PAPER_INPUT_SIZE = 256


class MoVQCenterCropBicubic:
    def __call__(self, image):
        width, height = image.size
        side = min(width, height)
        image = image.crop(((width-side)/2, (height-side)/2,
                            (width+side)/2, (height+side)/2))
        image = image.resize((256, 256), resample=Image.Resampling.BICUBIC, reducing_gap=1)
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 127.5 - 1
        return torch.from_numpy(np.ascontiguousarray(array.transpose(2, 0, 1)))


class VisionDataset(Dataset):
    def __init__(self, img_dir, transform=None, expected_count=None):
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.img_names = sorted(path.relative_to(self.img_dir).as_posix()
                                for path in self.img_dir.rglob("*")
                                if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"})
        if expected_count is not None and len(self.img_names) != expected_count:
            raise RuntimeError(f"{self.img_dir}: expected {expected_count} images, found {len(self.img_names)}")

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, index):
        with Image.open(self.img_dir / self.img_names[index]) as image:
            image = image.convert("RGB")
            return self.transform(image) if self.transform else image.copy()


paper_transform = MoVQCenterCropBicubic()
imagenet_dir = str(PREPARED_DATA["imagenet"])
google_earth_dir = str(PREPARED_DATA["gee_train"])
train_imagenet = VisionDataset(imagenet_dir, paper_transform, 10_000)
train_google_earth = VisionDataset(google_earth_dir, paper_transform, 1_000)
combined_train = ConcatDataset([train_imagenet, train_google_earth])
assert len(combined_train) == 11_000
DATA_MANIFEST = {
    "protocol": PAPER_DATA_PROTOCOL,
    "counts": {"imagenet": len(train_imagenet), "remote": len(train_google_earth), "total": len(combined_train)},
    "training_zip_sha256": {key: json.loads((PREPARED_DATA[key] / ".prepared.json").read_text())["zip_sha256"]
                            for key in ("imagenet", "gee_train")},
    "preprocessing": "largest-center-square/PIL-bicubic256-reducing_gap1/RGB-float32[-1,1]",
    "augmentation": "none",
    "remote_source": "GEE NAIP proxy; not author-released images",
    "evaluation_root": GEE_TEST_DIR,
}
print("Paper training pool: ImageNet=10000, Google Earth=1000, total=11000")


### Semantic model — paper-aligned Fig. 2 encoder + Fig. 3 MoVQ decoder

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

MODEL_VERSION = "fso_sc_paper_faithful_v6"
SEMANTIC_MODEL_CONFIG = {
    "group_norm_groups": 32,
    "activation": "SiLU",
    "downsample": "asymmetric_right_bottom_pad_then_stride2_conv",
    "upsample": "nearest_neighbor_then_conv",
    "loss_reduction": "mean",
    "input_preprocessing": "largest-center-square+bicubic256+RGB[-1,1]",
    "conv_initialization": "PyTorch_default",
    "codebook_initialization": "uniform(-1/K,1/K)",
    "decoder_attention_order": "VQGAN_inferred_interleaved_Res_Attn",
    "decoder_condition": "raw_Uq; post_quant_conv_only_for_decoder_feature",
}


def group_norm(channels, affine=True):
    return nn.GroupNorm(32, channels, eps=1e-6, affine=affine)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.norm1 = group_norm(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = group_norm(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.conv2(F.silu(self.norm2(h)))
        return self.shortcut(x) + h


class AttnBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = group_norm(channels)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h = self.norm(x)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class SpatialNorm(nn.Module):
    """Paper Eq. (25): gamma(U_q) * GN(F) + beta(U_q)."""

    def __init__(self, channels, latent_dim):
        super().__init__()
        # MoVQ replaces VQGAN's normalization with spatial conditioning while
        # retaining GroupNorm's learned affine parameters.
        self.norm = group_norm(channels, affine=True)
        self.conv_gamma = nn.Conv2d(latent_dim, channels, 1)
        self.conv_beta = nn.Conv2d(latent_dim, channels, 1)

    def forward(self, x, quantized_condition):
        if x.shape[2:] != quantized_condition.shape[2:]:
            quantized_condition = F.interpolate(
                quantized_condition, size=x.shape[2:], mode="nearest"
            )
        gamma = self.conv_gamma(quantized_condition)
        beta = self.conv_beta(quantized_condition)
        return gamma * self.norm(x) + beta


class SpatialResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, latent_dim=4):
        super().__init__()
        self.norm1 = SpatialNorm(in_channels, latent_dim)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = SpatialNorm(out_channels, latent_dim)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, quantized_condition):
        h = self.conv1(F.silu(self.norm1(x, quantized_condition)))
        h = self.conv2(F.silu(self.norm2(h, quantized_condition)))
        return self.shortcut(x) + h


class SpatialAttnBlock(nn.Module):
    def __init__(self, channels, latent_dim=4):
        super().__init__()
        self.norm = SpatialNorm(channels, latent_dim)
        self.q = nn.Conv2d(channels, channels, 1)
        self.k = nn.Conv2d(channels, channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x, quantized_condition):
        h = self.norm(x, quantized_condition)
        q, k, v = self.q(h), self.k(h), self.v(h)
        batch, channels, height, width = q.shape
        q = q.reshape(batch, channels, height * width).permute(0, 2, 1)
        k = k.reshape(batch, channels, height * width)
        attention = torch.bmm(q, k) * (channels ** -0.5)
        attention = F.softmax(attention, dim=2)
        v = v.reshape(batch, channels, height * width)
        h = torch.bmm(v, attention.permute(0, 2, 1)).reshape(
            batch, channels, height, width
        )
        return x + self.proj_out(h)


class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2.0, mode="nearest"))


class Downsample(nn.Module):
    """VQGAN downsample: asymmetric right/bottom pad then padding-free Conv."""

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=0)

    def forward(self, x):
        return self.conv(F.pad(x, (0, 1, 0, 1)))


class Encoder(nn.Module):
    def __init__(self, in_channels=3, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, 256, 3, padding=1)
        self.res1 = nn.Sequential(ResidualBlock(256, 256), ResidualBlock(256, 256))
        self.down1 = Downsample(256)
        self.res2 = nn.Sequential(ResidualBlock(256, 512), ResidualBlock(512, 512))
        self.down2 = Downsample(512)
        self.res3 = nn.Sequential(ResidualBlock(512, 512), ResidualBlock(512, 512))
        self.down3 = Downsample(512)
        self.res4 = nn.Sequential(ResidualBlock(512, 1024), ResidualBlock(1024, 1024))
        self.attn1 = nn.Sequential(AttnBlock(1024), AttnBlock(1024))
        self.res5 = ResidualBlock(1024, 1024)
        self.attn2 = AttnBlock(1024)
        self.res6 = ResidualBlock(1024, 1024)
        self.out_norm = group_norm(1024)
        self.out_conv = nn.Conv2d(1024, latent_dim, 3, padding=1)

    def forward(self, x):
        x = self.res1(self.in_conv(x))
        x = self.res2(self.down1(x))
        x = self.res3(self.down2(x))
        x = self.res4(self.down3(x))
        x = self.attn1(x)
        x = self.attn2(self.res5(x))
        return self.out_conv(F.silu(self.out_norm(self.res6(x))))


class Decoder(nn.Module):
    """Fig. 3 block set arranged in the dimensionally valid MoVQ/VQGAN order."""

    def __init__(self, latent_dim=4):
        super().__init__()
        self.in_conv = nn.Conv2d(latent_dim, 1024, 3, padding=1)

        # VQGAN mid block: Res -> Attn -> Res.
        self.mid_res1 = SpatialResBlock(1024, 1024, latent_dim)
        self.mid_attn = SpatialAttnBlock(1024, latent_dim)
        self.mid_res2 = SpatialResBlock(1024, 1024, latent_dim)

        # Deepest level: three Res+Attn pairs, then 32 -> 64.
        self.deep_res = nn.ModuleList([
            SpatialResBlock(1024, 1024, latent_dim) for _ in range(3)
        ])
        self.deep_attn = nn.ModuleList([
            SpatialAttnBlock(1024, latent_dim) for _ in range(3)
        ])
        self.up_1024 = Upsample(1024)

        # 64x64 level: 1024 -> 512 followed by two 512 blocks, then 64 -> 128.
        self.level_512a = nn.ModuleList([
            SpatialResBlock(1024, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
            SpatialResBlock(512, 512, latent_dim),
        ])
        self.up_512a = Upsample(512)

        # 128x128 level: three 512 blocks, then 128 -> 256.
        self.level_512b = nn.ModuleList([
            SpatialResBlock(512, 512, latent_dim) for _ in range(3)
        ])
        self.up_512b = Upsample(512)

        # 256x256 level: 512 -> 256 followed by two 256 blocks.
        self.level_256 = nn.ModuleList([
            SpatialResBlock(512, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
            SpatialResBlock(256, 256, latent_dim),
        ])
        self.out_norm = SpatialNorm(256, latent_dim)
        self.out_conv = nn.Conv2d(256, 3, 3, padding=1)

    def forward(self, decoder_input, quantized_condition):
        h = self.mid_res1(self.in_conv(decoder_input), quantized_condition)
        h = self.mid_attn(h, quantized_condition)
        h = self.mid_res2(h, quantized_condition)

        for res_block, attn_block in zip(self.deep_res, self.deep_attn):
            h = attn_block(
                res_block(h, quantized_condition), quantized_condition
            )
        h = self.up_1024(h)

        for block in self.level_512a:
            h = block(h, quantized_condition)
        h = self.up_512a(h)

        for block in self.level_512b:
            h = block(h, quantized_condition)
        h = self.up_512b(h)

        for block in self.level_256:
            h = block(h, quantized_condition)
        return self.out_conv(F.silu(self.out_norm(h, quantized_condition)))


class VectorQuantizer(nn.Module):
    def __init__(
        self,
        num_embeddings=16384,
        embedding_dim=4,
        commitment_cost=1.0,
        distance_chunk_size=2048,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        self.distance_chunk_size = distance_chunk_size
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(
            -1.0 / num_embeddings, 1.0 / num_embeddings
        )

    @torch.no_grad()
    def nearest_indices(self, flat_input):
        # VQ assignment remains FP32 even when the surrounding model uses AMP.
        vectors = flat_input.detach().float()
        codebook = self.embedding.weight.detach().float()
        codebook_norm = codebook.square().sum(dim=1).unsqueeze(0)
        nearest = []
        for chunk in vectors.split(self.distance_chunk_size, dim=0):
            distances = (
                chunk.square().sum(dim=1, keepdim=True)
                + codebook_norm
                - 2.0 * chunk @ codebook.t()
            )
            nearest.append(distances.argmin(dim=1))
        return torch.cat(nearest, dim=0)

    def lookup(self, encoding_indices, spatial_shape=None):
        if spatial_shape is None:
            if encoding_indices.ndim != 3:
                raise ValueError("indices must have shape [batch, height, width]")
            batch, height, width = encoding_indices.shape
        else:
            batch, height, width = spatial_shape
        flat_indices = encoding_indices.reshape(-1).long()
        quantized = F.embedding(flat_indices, self.embedding.weight)
        return quantized.view(batch, height, width, self.embedding_dim).permute(
            0, 3, 1, 2
        ).contiguous()

    def forward(self, inputs):
        channels_last = inputs.permute(0, 2, 3, 1).contiguous()
        batch, height, width, channels = channels_last.shape
        if channels != self.embedding_dim:
            raise ValueError(f"Expected embedding_dim={self.embedding_dim}, got {channels}")

        flat_input = channels_last.view(-1, self.embedding_dim)
        flat_indices = self.nearest_indices(flat_input)
        encoding_indices = flat_indices.view(batch, height, width)
        quantized = self.lookup(encoding_indices, (batch, height, width))

        encoder_loss = F.mse_loss(quantized.detach().float(), inputs.float())
        codebook_loss = F.mse_loss(quantized.float(), inputs.detach().float())
        vq_loss = codebook_loss + self.commitment_cost * encoder_loss

        quantized_st = inputs + (quantized.to(inputs.dtype) - inputs).detach()
        return quantized_st, vq_loss, encoding_indices


class FSO_VQVAE(nn.Module):
    def __init__(self, num_embeddings=16384, embedding_dim=4, commitment_cost=1.0):
        super().__init__()
        self.encoder = Encoder(in_channels=3, latent_dim=embedding_dim)
        self.pre_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.post_quant_conv = nn.Conv2d(embedding_dim, embedding_dim, 1)
        self.decoder = Decoder(latent_dim=embedding_dim)

    def encode(self, x):
        if x.ndim != 4 or x.shape[1] != 3:
            raise ValueError("Expected RGB input with shape [batch, 3, height, width]")
        if x.shape[-2] % 8 or x.shape[-1] % 8:
            raise ValueError("Input height and width must be divisible by 8")
        continuous = self.pre_quant_conv(self.encoder(x))
        quantized, vq_loss, indices = self.vq(continuous)
        return quantized, vq_loss, indices

    def decode_quantized(self, quantized):
        decoder_input = self.post_quant_conv(quantized)
        return self.decoder(decoder_input, quantized_condition=quantized)

    def decode_indices(self, indices, spatial_shape):
        return self.decode_quantized(self.vq.lookup(indices, spatial_shape))

    def forward(self, x):
        quantized, vq_loss, indices = self.encode(x)
        return self.decode_quantized(quantized), vq_loss, indices


# The architecture is intentionally not instantiated here; it is large (~hundreds of M parameters).

### 14-bit semantic index / QAM / FSO receiver path

In [ ]:
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F


BITS_PER_INDEX = 14
QAM_MAPPING_VERSION = "gray_qpsk_rect8qam_16qam_unit_energy_v1"
INTEGER_DTYPES = {torch.uint8, torch.int8, torch.int16, torch.int32, torch.int64}


def integers_to_bits(values, width):
    values = values.long()
    shifts = torch.arange(width - 1, -1, -1, device=values.device)
    return ((values.unsqueeze(-1) >> shifts) & 1).to(torch.uint8)


def bits_to_integers(bits):
    bits = bits.long()
    width = bits.shape[-1]
    weights = (2 ** torch.arange(width - 1, -1, -1, device=bits.device)).long()
    return (bits * weights).sum(dim=-1)


def indices_to_bits(indices, bits_per_index=BITS_PER_INDEX):
    if indices.dtype not in INTEGER_DTYPES:
        raise TypeError("indices must use an integer dtype")
    if indices.numel() and (
        (indices < 0).any().item() or (indices >= 2**bits_per_index).any().item()
    ):
        raise ValueError("Index is outside the fixed-width bit representation")
    return integers_to_bits(indices, bits_per_index)


def bits_to_indices(bits, bits_per_index=BITS_PER_INDEX):
    if bits.shape[-1] != bits_per_index:
        raise ValueError(f"Last dimension must be {bits_per_index} bits")
    if bits.dtype not in INTEGER_DTYPES and bits.dtype != torch.bool:
        raise TypeError("bits must use an integer or bool dtype")
    if bits.numel() and ((bits != 0) & (bits != 1)).any().item():
        raise ValueError("bits must contain only 0 and 1")
    return bits_to_integers(bits)


def qam_constellation(modulation, device):
    modulation = modulation.upper()
    supported = {"QPSK": 2, "8QAM": 3, "16QAM": 4}
    if modulation not in supported:
        raise ValueError(f"Unsupported modulation: {modulation}")
    bits_per_symbol = supported[modulation]
    labels = integers_to_bits(
        torch.arange(2**bits_per_symbol, device=device), bits_per_symbol
    ).float()

    if modulation == "QPSK":
        i = 1.0 - 2.0 * labels[:, 0]
        q = 1.0 - 2.0 * labels[:, 1]
        normalization = math.sqrt(2.0)
    elif modulation == "8QAM":
        # Rectangular Gray-labelled 8-QAM. The paper does not specify its 8-QAM geometry.
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        i = gray_levels[i_code]
        q = 2.0 * labels[:, 2] - 1.0
        normalization = math.sqrt(6.0)
    else:
        gray_levels = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=device)
        i_code = (2 * labels[:, 0] + labels[:, 1]).long()
        q_code = (2 * labels[:, 2] + labels[:, 3]).long()
        i = gray_levels[i_code]
        q = gray_levels[q_code]
        normalization = math.sqrt(10.0)

    points = torch.complex(i, q) / normalization
    return points.to(torch.complex64), labels.to(torch.uint8), bits_per_symbol


def qam_modulate(bits, modulation):
    points, _, bits_per_symbol = qam_constellation(modulation, bits.device)
    if bits.shape[-1] != bits_per_symbol:
        raise ValueError(f"{modulation} expects {bits_per_symbol} bits per symbol")
    symbol_ids = bits_to_integers(bits).long()
    return points[symbol_ids]


def qam_demodulate(symbols, modulation):
    points, labels, _ = qam_constellation(modulation, symbols.device)
    # Cartesian-product QAM: limiting observations outside the outer levels
    # preserves exact nearest-point decisions and avoids large-float cancellation.
    i_limit = 2 * points.real.abs().max()
    q_limit = 2 * points.imag.abs().max()
    stable = torch.complex(symbols.real.clamp(-i_limit, i_limit),
                           symbols.imag.clamp(-q_limit, q_limit))
    distances = (stable.reshape(-1, 1) - points.reshape(1, -1)).abs().square()
    nearest = distances.argmin(dim=1)
    return labels[nearest].view(*symbols.shape, labels.shape[-1])


@dataclass(frozen=True)
class FSOChannelOutput:
    """Receiver observation before hard QAM decision.

    Return `is_equalized=True` after coherent receiver DSP, or provide the
    complex channel gain so this link can perform one-tap equalization.
    """

    symbols: torch.Tensor
    channel_gain: object = None
    is_equalized: bool = False
    diagnostics: object = None


class IdentityFSOChannel:
    """No-impairment receiver hook used to test the digital path."""

    def __call__(self, symbols, **channel_kwargs):
        return FSOChannelOutput(symbols=symbols, is_equalized=True)


def equalize_fso_output(channel_output, transmitted):
    if isinstance(channel_output, FSOChannelOutput):
        received = channel_output.symbols
        channel_gain = channel_output.channel_gain
        is_equalized = channel_output.is_equalized
    elif isinstance(channel_output, tuple) and len(channel_output) == 2:
        received, channel_gain = channel_output
        is_equalized = False
    else:
        raise TypeError(
            "FSO receiver must return FSOChannelOutput or (symbols, channel_gain)"
        )

    if not torch.is_tensor(received):
        raise TypeError("FSO receiver must return a torch.Tensor observation")
    if received.device != transmitted.device:
        raise ValueError(
            f"FSO receiver changed device: {transmitted.device} -> {received.device}"
        )
    if not torch.is_complex(received):
        raise TypeError("FSO receiver output must be complex baseband symbols")
    if received.shape != transmitted.shape:
        raise ValueError(
            f"FSO receiver changed symbol shape: {transmitted.shape} -> {received.shape}"
        )

    if not is_equalized:
        if not torch.is_tensor(channel_gain):
            raise TypeError("Unequalized FSO output must include a tensor channel_gain")
        if channel_gain.device != received.device:
            raise ValueError("channel_gain and received symbols must share a device")
        try:
            channel_gain = torch.broadcast_to(channel_gain, received.shape)
        except RuntimeError as error:
            raise ValueError("channel_gain is not broadcastable to the symbol shape") from error
        if (channel_gain.abs() == 0).any().item():
            raise ValueError("channel_gain contains an exact zero")
        if not torch.isfinite(channel_gain).all().item():
            raise ValueError("channel_gain contains NaN or Inf")
        denominator = channel_gain.abs().square().clamp_min(
            torch.finfo(received.real.dtype).tiny
        )
        received = received * channel_gain.conj() / denominator

    if not torch.isfinite(received).all().item():
        raise ValueError("FSO receiver output contains NaN or Inf")
    return received


@dataclass
class QAMIndexLink:
    """14-bit digital link ending in equalized complex [B, S] symbols."""

    fso_channel: object
    modulation: str = "QPSK"
    bits_per_index: int = BITS_PER_INDEX
    receiver_config: object = None

    def reproducibility_config(self):
        if self.receiver_config is not None and not isinstance(
            self.receiver_config, dict
        ):
            raise TypeError("receiver_config must be a serializable dict or None")
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulation": self.modulation.upper(),
            "bits_per_index": self.bits_per_index,
            "receiver_config": self.receiver_config,
        }

    @torch.no_grad()
    def __call__(self, indices, return_statistics=False, **channel_kwargs):
        if indices.ndim < 2:
            raise ValueError("indices must retain a batch dimension")
        original_shape = indices.shape
        batch = original_shape[0]
        valid_bits_per_sample = indices[0].numel() * self.bits_per_index
        bitstream = indices_to_bits(indices, self.bits_per_index).reshape(batch, -1)
        _, _, bits_per_symbol = qam_constellation(self.modulation, indices.device)
        padding = (-valid_bits_per_sample) % bits_per_symbol
        if padding:
            bitstream = F.pad(bitstream, (0, padding))

        grouped_bits = bitstream.view(batch, -1, bits_per_symbol)
        transmitted = qam_modulate(grouped_bits, self.modulation)
        channel_output = self.fso_channel(transmitted, **channel_kwargs)
        received = equalize_fso_output(channel_output, transmitted)
        recovered_grouped = qam_demodulate(received, self.modulation)
        recovered = recovered_grouped.reshape(batch, -1)
        valid_mask = torch.arange(
            recovered.shape[1], device=indices.device
        ).view(1, -1) < valid_bits_per_sample
        transmitted_bit_error_mask = recovered != bitstream
        data_bit_error_mask = transmitted_bit_error_mask & valid_mask
        # SER is defined over every transmitted QAM symbol, including the final
        # padded symbol. Payload BER/TER continue to exclude padding bits.
        symbol_error_mask = transmitted_bit_error_mask.view(
            batch, -1, bits_per_symbol
        ).any(dim=-1)
        if padding:
            recovered = recovered[:, :-padding]
        recovered = recovered.view(*original_shape, self.bits_per_index)
        recovered_indices = bits_to_indices(
            recovered, self.bits_per_index
        ).view(original_shape)
        if not return_statistics:
            return recovered_indices

        bit_errors = int(data_bit_error_mask.sum().item())
        valid_bits = batch * valid_bits_per_sample
        transmitted_bit_errors = int(transmitted_bit_error_mask.sum().item())
        transmitted_bits = bitstream.numel()
        symbol_errors = int(symbol_error_mask.sum().item())
        valid_symbols = symbol_error_mask.numel()
        index_errors = int((recovered_indices != indices).sum().item())
        statistics = {
            "modulation": self.modulation.upper(),
            "padding_bits_per_sample": padding,
            "bit_errors": bit_errors,
            "valid_bits": valid_bits,
            "ber": bit_errors / max(valid_bits, 1),
            "transmitted_bit_errors": transmitted_bit_errors,
            "transmitted_bits": transmitted_bits,
            "transmitted_ber": transmitted_bit_errors / max(transmitted_bits, 1),
            "symbol_errors": symbol_errors,
            "valid_symbols": valid_symbols,
            "ser": symbol_errors / max(valid_symbols, 1),
            "index_errors": index_errors,
            "indices": indices.numel(),
            "ter": index_errors / max(indices.numel(), 1),
        }
        if isinstance(channel_output, FSOChannelOutput) and channel_output.diagnostics:
            statistics["channel"] = channel_output.diagnostics
        return recovered_indices, statistics


def qam_identity_roundtrip_test():
    test_indices = torch.tensor([
        [0, 1, 2, 8191, 8192, 16383, 77],
        [77, 16383, 8192, 8191, 2, 1, 0],
    ]).view(2, 1, 7)
    for modulation in ("QPSK", "8QAM", "16QAM"):
        link = QAMIndexLink(IdentityFSOChannel(), modulation=modulation)
        recovered = link(test_indices)
        assert torch.equal(recovered, test_indices), f"{modulation} round trip failed"
    print("Identity-channel index/QAM round-trip passed: QPSK, 8QAM, 16QAM")


qam_identity_roundtrip_test()

### Paper Eq. (3)–(18) Málaga satellite-to-ground FSO channel

In [ ]:
"""Paper-parameter symbol-rate FSO channel with explicit heterodyne shot floor.

The vacuum/shot term is a physically motivated missing receiver detail, not an
explicit Chen et al. parameter or an LPIPS-curve calibration. One complex
heterodyne mode/symbol has E|n|^2=h*nu/eta, so the optical-field (sqrt W)
convention gives N_complex=(q_e/R)*R_s W; I and Q each have half that variance.
Reference: A. Martinez, Physical Review A 77,032116 (2008),
doi:10.1103/PhysRevA.77.032116, https://ir.cwi.nl/pub/13524/13524D.pdf .
"""

from dataclasses import asdict, dataclass, replace
import math
import warnings

import numpy as np
import torch


PAPER_FSO_CHANNEL_VERSION = "chen2025_eq3_18_heterodyne_shot_v6"
WEATHER_ATTENUATION_DB_PER_KM = {
    "clear": 0.147,
    "dust": 1.043,
    "fog": 33.979,
    "rain": 1.403,
    "snow": 2.145,
}

# The paper specifies FOV=0.1 but does not print a unit, and it does not give
# the optical filter width or the proprietary VPI receiver/DSP implementation.
# These assumptions are explicit and stored with the final model metadata.
PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS = {
    "fov_interpretation": "0.1 rad",
    "optical_filter_bandwidth": "1 nm",
    "background_noise_conversion": (
        "Eq.-(4) optical background power is used as unit-energy complex "
        "matched-filter noise variance; proprietary coherent receiver is undisclosed"
    ),
    "ase_model": (
        "Table-I launch power is post-transmitter-EDFA; standard two-polarization "
        "EDFA ASE is propagated through the optical path"
    ),
    "ase_noise_bandwidth": "28 GHz symbol-rate equivalent",
    "diversity_ase_correlation": "one common transmitter-ASE waveform across Rx branches",
    "photodiode_responsivity": (
        "Table-I R=0.85 A/W determines eta=R*h*nu/q_e; "
        "one-complex-mode/symbol heterodyne shot floor q_e*Rs/R is input-referred; "
        "strong LO and negligible receiver thermal/RIN/dark current are assumptions"
    ),
    "receiver_shot_noise": (
        "complex heterodyne mode variance h*nu/eta per symbol; "
        "N_complex=q_e*Rs/R W, with half per real quadrature; "
        "physically motivated undisclosed implementation detail, not curve-fitted"
    ),
    "receiver_shot_noise_reference": "https://doi.org/10.1103/PhysRevA.77.032116",
    "pulse_shaping": "symbol-rate matched-filter equivalent; RRC rolloff undisclosed",
    "carrier_recovery": (
        "raw channel returns true complex gain for an explicitly ideal/oracle baseline; "
        "the separate SignalDSP wrapper uses only block magnitude and estimates phase/frequency"
    ),
    "rytov_quadrature": "xi=u^2 ground-clustered trapezoid with convergence self-test",
    "malaga_phase_difference": "pi/2 chosen phase assumption; paper undisclosed",
    "malaga_beta_fit": (
        "paper omits beta and phase; nearest Eq.-(15) natural beta is used with "
        "an explicit feasibility/residual diagnostic (strict policy is available)"
    ),
    "fading_coherence": "one independent block fade per image and receiver",
    "8qam_geometry": "rectangular Gray 8-QAM; paper only states 8-QAM",
}


def paper_generator_device_matches(generator, tensor_device):
    generator_device = generator.device
    if generator_device.type != tensor_device.type:
        return False
    if tensor_device.type != "cuda":
        return generator_device == tensor_device
    generator_index = (
        torch.cuda.current_device()
        if generator_device.index is None
        else generator_device.index
    )
    tensor_index = (
        torch.cuda.current_device() if tensor_device.index is None else tensor_device.index
    )
    return generator_index == tensor_index


@dataclass(frozen=True)
class PaperFSOSystemConfig:
    # Paper Table I / Eqs. (4)-(18).
    wavelength_m: float = 1550e-9
    satellite_altitude_m: float = 620e3
    receiver_altitude_m: float = 0.0
    earth_radius_m: float = 6371e3
    earth_gravitational_parameter: float = 3.986004418e14
    light_speed_m_s: float = 299792458.0
    planck_constant_j_s: float = 6.62607015e-34
    electron_charge_c: float = 1.602176634e-19
    beam_radius_m: float = 0.05
    receiver_aperture_m: float = 0.318
    rms_wind_speed_m_s: float = 21.0
    ground_cn2_m_neg_two_thirds: float = 1.7e-14
    atmospheric_range_m: float = 20e3
    fog_range_m: float = 8e3
    field_of_view_rad: float = 0.1
    optical_filter_bandwidth_um: float = 1e-3  # 1 nm; not disclosed.
    symbol_rate_baud: float = 28e9
    amplifier_gain_db: float = 30.0
    amplifier_noise_figure_db: float = 4.0
    lo_linewidth_hz: float = 1000.0
    photodiode_responsivity_a_w: float = 0.85
    doppler_residual_ratio: float = 0.1
    malaga_omega: float = 0.5
    malaga_b0: float = 0.25
    malaga_phase_difference_rad: float = math.pi / 2
    malaga_beta: object = None
    max_derived_beta: int = 1000
    beta_fit_policy: str = "nearest_warn"
    beta_fit_relative_tolerance: float = 1e-3
    integration_points: int = 4097
    integration_grid_power: float = 2.0
    num_receivers: int = 1
    combining: str = "MRC"
    include_turbulence: bool = True
    include_background_noise: bool = True
    include_ase_noise: bool = True
    include_shot_noise: bool = True
    heterodyne_shot_noise_convention: float = 1.0
    include_doppler: bool = True
    include_phase_noise: bool = True

    # Convenience/reference condition; this combination is not a paper-wide default.
    launch_power_dbm: float = 33.0
    zenith_angle_deg: float = 30.0
    pointing_error_urad: float = 20.0
    scattered_power_ratio: float = 0.95
    background_light_radiance_w_m2_sr_um: float = 1e-4
    weather: str = "clear"

    def __post_init__(self):
        if self.integration_points < 257 or self.integration_points % 2 == 0:
            raise ValueError("integration_points must be an odd integer >= 257")
        if self.integration_grid_power < 1.0:
            raise ValueError("integration_grid_power must be at least 1")
        if self.photodiode_responsivity_a_w <= 0:
            raise ValueError("Photodiode responsivity must be positive")
        quantum_efficiency = (
            self.photodiode_responsivity_a_w * self.planck_constant_j_s
            * self.light_speed_m_s / self.wavelength_m / self.electron_charge_c
        )
        if not 0 < quantum_efficiency <= 1:
            raise ValueError("Responsivity implies unphysical photodiode quantum efficiency")
        if self.heterodyne_shot_noise_convention <= 0:
            raise ValueError("Heterodyne complex-noise convention factor must be positive")
        if self.num_receivers < 1:
            raise ValueError("num_receivers must be positive")
        if self.combining.upper() not in {"MRC", "EGC"}:
            raise ValueError("combining must be MRC or EGC")
        if not math.isclose(self.malaga_omega + 2 * self.malaga_b0, 1.0):
            raise ValueError("The paper normalizes Malaga power as Omega + 2*b0 = 1")
        if self.beta_fit_policy not in {"nearest_warn", "strict"}:
            raise ValueError("beta_fit_policy must be nearest_warn or strict")
        if self.beta_fit_relative_tolerance < 0:
            raise ValueError("beta_fit_relative_tolerance must be nonnegative")


def _trapezoid(values, grid):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(values, grid)
    return np.trapz(values, grid)


def hufnagel_valley_cn2(altitude_m, config):
    h = np.asarray(altitude_m, dtype=np.float64)
    v = config.rms_wind_speed_m_s
    a = config.ground_cn2_m_neg_two_thirds
    return (
        0.00594 * (v / 27.0) ** 2 * (1e-5 * h) ** 10 * np.exp(-h / 1000.0)
        + 2.7e-16 * np.exp(-h / 1500.0)
        + a * np.exp(-h / 100.0)
    )


def satellite_slant_range_m(zenith_angle_deg, config):
    phi = math.radians(float(zenith_angle_deg))
    if not 0.0 <= phi < math.pi / 2:
        raise ValueError("zenith_angle_deg must be in [0, 90)")
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    satellite_radius = config.earth_radius_m + config.satellite_altitude_m
    return (
        math.sqrt(
            satellite_radius**2
            - ground_radius**2 * math.sin(phi) ** 2
        )
        - ground_radius * math.cos(phi)
    )


def path_altitude_m(path_fraction, slant_range_m, zenith_angle_deg, config):
    xi = np.asarray(path_fraction, dtype=np.float64)
    phi = math.radians(float(zenith_angle_deg))
    ground_radius = config.earth_radius_m + config.receiver_altitude_m
    distance = xi * slant_range_m
    radius = np.sqrt(
        ground_radius**2
        + distance**2
        + 2.0 * ground_radius * distance * math.cos(phi)
    )
    return radius - config.earth_radius_m


def shadowed_rician_second_moment(beta, omega_prime, gamma):
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    return (
        beta_independent + omega_prime**2 / float(beta)
    ) / mean_y**2


def fit_malaga_beta_eq15(target_second_moment, omega_prime, gamma, max_beta):
    """Nearest natural beta plus an honest Eq.-(15) feasibility diagnostic."""
    mean_y = omega_prime + gamma
    beta_independent = omega_prime**2 + 4 * omega_prime * gamma + 2 * gamma**2
    denominator = target_second_moment * mean_y**2 - beta_independent
    continuous_beta = (
        omega_prime**2 / denominator if denominator > 0 else float("inf")
    )
    continuum_feasible = 1.0 <= continuous_beta <= float(max_beta)
    if math.isfinite(continuous_beta):
        candidates = {
            1,
            int(max_beta),
            min(max(int(math.floor(continuous_beta)), 1), int(max_beta)),
            min(max(int(math.ceil(continuous_beta)), 1), int(max_beta)),
        }
    else:
        candidates = {int(max_beta)}
    beta = min(
        candidates,
        key=lambda value: abs(
            shadowed_rician_second_moment(value, omega_prime, gamma)
            - target_second_moment
        ),
    )
    fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
    relative_residual = abs(fitted - target_second_moment) / max(
        abs(target_second_moment), 1e-30
    )
    return {
        "beta": int(beta),
        "continuous_beta": continuous_beta,
        "continuum_feasible_within_search": continuum_feasible,
        "fitted_second_moment": fitted,
        "relative_residual": relative_residual,
        "minimum_second_moment": beta_independent / mean_y**2,
        "maximum_natural_second_moment": shadowed_rician_second_moment(
            1, omega_prime, gamma
        ),
    }


def compute_paper_link_budget(
    config,
    launch_power_dbm=None,
    zenith_angle_deg=None,
    pointing_error_urad=None,
    scattered_power_ratio=None,
    background_light_radiance_w_m2_sr_um=None,
    weather=None,
):
    """Implement paper Eqs. (4)-(18) and Table I as scalar diagnostics."""
    launch_power_dbm = (
        config.launch_power_dbm if launch_power_dbm is None else float(launch_power_dbm)
    )
    zenith_angle_deg = (
        config.zenith_angle_deg if zenith_angle_deg is None else float(zenith_angle_deg)
    )
    pointing_error_urad = (
        config.pointing_error_urad
        if pointing_error_urad is None
        else float(pointing_error_urad)
    )
    rho = (
        config.scattered_power_ratio
        if scattered_power_ratio is None
        else float(scattered_power_ratio)
    )
    radiance = (
        config.background_light_radiance_w_m2_sr_um
        if background_light_radiance_w_m2_sr_um is None
        else float(background_light_radiance_w_m2_sr_um)
    )
    weather = (config.weather if weather is None else weather).lower()
    if weather not in WEATHER_ATTENUATION_DB_PER_KM:
        raise ValueError(f"Unsupported weather: {weather}")
    if not 0.0 <= rho <= 1.0:
        raise ValueError("scattered_power_ratio must be in [0, 1]")
    if radiance < 0.0:
        raise ValueError("background light radiance cannot be negative")

    wavelength = config.wavelength_m
    wavenumber = 2.0 * math.pi / wavelength
    slant_range = satellite_slant_range_m(zenith_angle_deg, config)

    theta0 = 1.0  # Collimated input beam: F0 -> infinity.
    lambda0 = 2.0 * slant_range / (wavenumber * config.beam_radius_m**2)
    denominator = theta0**2 + lambda0**2
    theta1 = theta0 / denominator
    lambda1 = lambda0 / denominator
    diffraction_radius = config.beam_radius_m / math.sqrt(theta1**2 + lambda1**2)

    # The Hufnagel-Valley ground term decays on a ~100-m altitude scale. A
    # uniform 4097-point slant grid undersamples it, so cluster samples near xi=0.
    xi = np.linspace(
        0.0, 1.0, config.integration_points, dtype=np.float64
    ) ** config.integration_grid_power
    altitude = path_altitude_m(xi, slant_range, zenith_angle_deg, config)
    cn2 = hufnagel_valley_cn2(altitude, config)
    mu_2d = float(_trapezoid(cn2 * xi ** (5.0 / 3.0), xi))
    spreading_factor = (
        4.35
        * mu_2d
        * lambda1 ** (5.0 / 6.0)
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
    )
    long_term_radius = diffraction_radius * math.sqrt(1.0 + spreading_factor)

    geometric_scintillation_db = -10.0 * math.log10(
        config.receiver_aperture_m**2 / (2.0 * long_term_radius**2)
    )
    pointing_offset_m = slant_range * pointing_error_urad * 1e-6
    pointing_db = (10.0 / math.log(10.0)) * (
        2.0 * pointing_offset_m**2 / long_term_radius**2
    )
    weather_range_m = config.fog_range_m if weather == "fog" else config.atmospheric_range_m
    atmospheric_db = WEATHER_ATTENUATION_DB_PER_KM[weather] * weather_range_m / 1000.0
    total_attenuation_db = geometric_scintillation_db + pointing_db + atmospheric_db

    complex_kernel = (
        xi ** (5.0 / 6.0)
        * (
            lambda1 * xi
            + 1j * (1.0 - (1.0 - theta1) * xi)
        ) ** (5.0 / 6.0)
        - lambda1 ** (5.0 / 6.0) * xi ** (5.0 / 3.0)
    )
    rytov_variance = float(
        8.7
        * wavenumber ** (7.0 / 6.0)
        * slant_range ** (11.0 / 6.0)
        * np.real(_trapezoid(cn2 * complex_kernel, xi))
    )
    rytov_variance = max(rytov_variance, 0.0)
    sigma_lnx2 = (
        0.49
        * rytov_variance
        / (
            1.0
            + (1.0 + theta1) * 0.56 * rytov_variance ** (6.0 / 5.0)
        ) ** (7.0 / 6.0)
    )
    sigma_lny2 = (
        0.51
        * rytov_variance
        / (1.0 + 0.69 * rytov_variance ** (6.0 / 5.0)) ** (5.0 / 6.0)
    )
    alpha = 1.0 / math.expm1(max(sigma_lnx2, 1e-12))

    omega = config.malaga_omega
    b0 = config.malaga_b0
    phase_difference = config.malaga_phase_difference_rad
    omega_prime = (
        omega
        + 2.0 * b0 * rho
        + 2.0 * math.sqrt(2.0 * b0 * omega * rho) * math.cos(phase_difference)
    )
    gamma = 2.0 * b0 * (1.0 - rho)
    target_y_second_moment = math.exp(sigma_lny2)
    if config.malaga_beta is None:
        beta_fit = fit_malaga_beta_eq15(
            target_y_second_moment,
            omega_prime,
            gamma,
            config.max_derived_beta,
        )
        beta = beta_fit["beta"]
        beta_fit_feasible = (
            beta_fit["continuum_feasible_within_search"]
            and beta_fit["relative_residual"]
            <= config.beta_fit_relative_tolerance
        )
        beta_source = "eq15_nearest_natural"
        if config.beta_fit_policy == "strict" and not beta_fit_feasible:
            raise ValueError(
                "Paper-undisclosed Malaga beta/phase cannot satisfy Eq. (15) "
                f"within tolerance: target={target_y_second_moment:.8g}, "
                f"nearest={beta_fit['fitted_second_moment']:.8g}, "
                f"relative residual={beta_fit['relative_residual']:.3%}"
            )
    else:
        beta = int(config.malaga_beta)
        if beta < 1:
            raise ValueError("malaga_beta must be a natural number")
        fitted = shadowed_rician_second_moment(beta, omega_prime, gamma)
        beta_fit = {
            "beta": beta,
            "continuous_beta": None,
            "continuum_feasible_within_search": None,
            "fitted_second_moment": fitted,
            "relative_residual": abs(fitted - target_y_second_moment)
            / max(abs(target_y_second_moment), 1e-30),
            "minimum_second_moment": None,
            "maximum_natural_second_moment": None,
        }
        beta_fit_feasible = None
        beta_source = "explicit_user_assumption"
    fitted_y_second_moment = beta_fit["fitted_second_moment"]

    earth_radius = config.earth_radius_m
    orbit_radius = earth_radius + config.satellite_altitude_m
    orbital_angular_velocity = math.sqrt(
        config.earth_gravitational_parameter / orbit_radius**3
    )
    phi = math.radians(zenith_angle_deg)
    phi_max = math.pi / 2.0
    eta = math.cos(
        math.acos(earth_radius / orbit_radius * math.cos(phi_max)) - phi_max
    )
    optical_frequency_hz = config.light_speed_m_s / wavelength
    doppler_denominator = math.sqrt(
        earth_radius**2
        + orbit_radius**2
        - 2.0 * earth_radius * orbit_radius * math.cos(phi) * eta
    )
    doppler_hz = -(
        optical_frequency_hz
        / config.light_speed_m_s
        * orbital_angular_velocity
        * earth_radius
        * orbit_radius
        * math.sin(phi)
        * eta
        / doppler_denominator
    )
    residual_doppler_hz = config.doppler_residual_ratio * doppler_hz

    background_noise_w = (
        radiance
        * math.pi**2
        * config.field_of_view_rad**2
        * config.receiver_aperture_m**2
        / 16.0
        * config.optical_filter_bandwidth_um
    )
    amplifier_gain_linear = 10.0 ** (config.amplifier_gain_db / 10.0)
    noise_factor = 10.0 ** (config.amplifier_noise_figure_db / 10.0)
    spontaneous_emission_factor = (
        noise_factor - 1.0 / amplifier_gain_linear
    ) / (2.0 * (1.0 - 1.0 / amplifier_gain_linear))
    ase_noise_w = (
        2.0
        * spontaneous_emission_factor
        * config.planck_constant_j_s
        * optical_frequency_hz
        * (amplifier_gain_linear - 1.0)
        * config.symbol_rate_baud
    )

    # One detected complex temporal mode per Nyquist symbol. Equivalent
    # input field noise is independent between receiver branches, not common
    # transmitter ASE. No unknown LO power or arbitrary thermal floor is fitted.
    quantum_efficiency = (
        config.photodiode_responsivity_a_w * config.planck_constant_j_s
        * optical_frequency_hz / config.electron_charge_c
    )
    receiver_shot_noise_w = (
        config.heterodyne_shot_noise_convention * config.electron_charge_c
        / config.photodiode_responsivity_a_w * config.symbol_rate_baud
    )

    return {
        "channel_version": PAPER_FSO_CHANNEL_VERSION,
        "launch_power_dbm": launch_power_dbm,
        "launch_power_w": 10.0 ** ((launch_power_dbm - 30.0) / 10.0),
        "zenith_angle_deg": zenith_angle_deg,
        "pointing_error_urad": pointing_error_urad,
        "scattered_power_ratio": rho,
        "background_light_radiance_w_m2_sr_um": radiance,
        "weather": weather,
        "slant_range_m": slant_range,
        "theta1": theta1,
        "lambda1": lambda1,
        "diffraction_radius_m": diffraction_radius,
        "long_term_radius_m": long_term_radius,
        "geometric_scintillation_db": geometric_scintillation_db,
        "pointing_db": pointing_db,
        "atmospheric_db": atmospheric_db,
        "total_attenuation_db": total_attenuation_db,
        "rytov_variance": rytov_variance,
        "sigma_lnx2": sigma_lnx2,
        "sigma_lny2": sigma_lny2,
        "malaga_alpha": alpha,
        "malaga_beta": beta,
        "malaga_beta_source": beta_source,
        "malaga_beta_fit_feasible": beta_fit_feasible,
        "malaga_beta_continuous": beta_fit["continuous_beta"],
        "malaga_beta_fit_relative_residual": beta_fit["relative_residual"],
        "malaga_beta_minimum_second_moment": beta_fit["minimum_second_moment"],
        "malaga_beta_maximum_natural_second_moment": beta_fit[
            "maximum_natural_second_moment"
        ],
        "malaga_omega_prime": omega_prime,
        "malaga_gamma": gamma,
        "malaga_mean_y": omega_prime + gamma,
        "target_y_second_moment": target_y_second_moment,
        "fitted_y_second_moment": fitted_y_second_moment,
        "doppler_hz": doppler_hz,
        "residual_doppler_hz": residual_doppler_hz,
        "background_noise_w": background_noise_w,
        "amplifier_gain_linear": amplifier_gain_linear,
        "ase_noise_w": ase_noise_w,
        "photodiode_quantum_efficiency": quantum_efficiency,
        "receiver_shot_noise_w": receiver_shot_noise_w,
        "complex_shot_noise_convention": config.heterodyne_shot_noise_convention,
    }


def _real_dtype(complex_dtype):
    if complex_dtype == torch.complex128:
        return torch.float64
    return torch.float32


def _complex_normal(shape, device, complex_dtype, generator=None):
    real_dtype = _real_dtype(complex_dtype)
    real = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    imag = torch.randn(shape, device=device, dtype=real_dtype, generator=generator)
    return torch.complex(real, imag) / math.sqrt(2.0)


def _gamma_unit_mean(shape_parameter, shape, device, dtype, generator=None):
    concentration = torch.full(
        shape,
        float(shape_parameter),
        device=device,
        dtype=dtype,
    )
    # PyTorch distributions do not expose a generator argument. This native
    # primitive does and is available in the paper's PyTorch 2.0 runtime.
    return torch._standard_gamma(concentration, generator=generator) / float(
        shape_parameter
    )


class PaperMalagaFSOChannel:
    """Paper Eq. (3) channel with Eqs. (4)-(18) and ideal coherent DSP.

    This is a symbol-rate matched-filter equivalent. Block Malaga fading is
    sampled independently per image/receiver. Doppler and LO phase are applied
    before ideal CFR/CPR/equalization through the returned complex channel gain.
    """

    def __init__(self, config=None):
        self.config = config or PaperFSOSystemConfig()
        self._budget_cache = {}
        self._reproducibility_config_cache = None
        self._warned_infeasible_beta_fit = False

    def reproducibility_config(self):
        if self._reproducibility_config_cache is None:
            reference = compute_paper_link_budget(self.config)
            self._reproducibility_config_cache = {
                "channel_version": PAPER_FSO_CHANNEL_VERSION,
                "system": asdict(self.config),
                "undisclosed_assumptions": PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS,
                "reference_beta_diagnostic": {
                    "beta": reference["malaga_beta"],
                    "source": reference["malaga_beta_source"],
                    "fit_feasible": reference["malaga_beta_fit_feasible"],
                    "target_second_moment": round(
                        reference["target_y_second_moment"], 12
                    ),
                    "fitted_second_moment": round(
                        reference["fitted_y_second_moment"], 12
                    ),
                    "relative_residual": round(
                        reference["malaga_beta_fit_relative_residual"], 12
                    ),
                },
            }
        return self._reproducibility_config_cache

    def _budget(self, **condition):
        key = tuple(sorted(condition.items()))
        if key not in self._budget_cache:
            self._budget_cache[key] = compute_paper_link_budget(
                self.config, **condition
            )
        budget = self._budget_cache[key]
        if (
            self.config.malaga_beta is None
            and self.config.beta_fit_policy == "nearest_warn"
            and not budget["malaga_beta_fit_feasible"]
            and not self._warned_infeasible_beta_fit
        ):
            warnings.warn(
                "The paper omits Malaga beta and phase, and Eq. (15) has no "
                "fit within the configured range/tolerance for this condition; "
                f"using beta={budget['malaga_beta']} with relative moment "
                f"residual={budget['malaga_beta_fit_relative_residual']:.3%}. "
                "Set an explicit malaga_beta or beta_fit_policy='strict'.",
                RuntimeWarning,
            )
            self._warned_infeasible_beta_fit = True
        return budget

    def sample_malaga_power(self, shape, budget, device, dtype, generator=None):
        if not self.config.include_turbulence:
            return torch.ones(shape, device=device, dtype=dtype)
        large_scale = _gamma_unit_mean(
            budget["malaga_alpha"], shape, device, dtype, generator
        )
        shadowing = _gamma_unit_mean(
            budget["malaga_beta"], shape, device, dtype, generator
        )
        rho = budget["scattered_power_ratio"]
        coherent_phasor = (
            math.sqrt(self.config.malaga_omega)
            + math.sqrt(2.0 * self.config.malaga_b0 * rho)
            * complex(
                math.cos(-self.config.malaga_phase_difference_rad),
                math.sin(-self.config.malaga_phase_difference_rad),
            )
        )
        diffuse = _complex_normal(
            shape,
            device,
            torch.complex128 if dtype == torch.float64 else torch.complex64,
            generator,
        ) * math.sqrt(budget["malaga_gamma"])
        small_field = shadowing.sqrt().to(diffuse.dtype) * coherent_phasor + diffuse
        small_power = small_field.abs().square().to(dtype)
        return large_scale * small_power / budget["malaga_mean_y"]

    @torch.no_grad()
    def __call__(
        self,
        symbols,
        launch_power_dbm=None,
        zenith_angle_deg=None,
        pointing_error_urad=None,
        scattered_power_ratio=None,
        background_light_radiance_w_m2_sr_um=None,
        weather=None,
        generator=None,
    ):
        if symbols.ndim != 2 or not torch.is_complex(symbols):
            raise ValueError("symbols must be complex baseband with shape [batch, symbols]")
        if generator is not None and not paper_generator_device_matches(
            generator, symbols.device
        ):
            raise ValueError("generator and symbols must use the same device")

        condition = {
            "launch_power_dbm": (
                self.config.launch_power_dbm
                if launch_power_dbm is None
                else float(launch_power_dbm)
            ),
            "zenith_angle_deg": (
                self.config.zenith_angle_deg
                if zenith_angle_deg is None
                else float(zenith_angle_deg)
            ),
            "pointing_error_urad": (
                self.config.pointing_error_urad
                if pointing_error_urad is None
                else float(pointing_error_urad)
            ),
            "scattered_power_ratio": (
                self.config.scattered_power_ratio
                if scattered_power_ratio is None
                else float(scattered_power_ratio)
            ),
            "background_light_radiance_w_m2_sr_um": (
                self.config.background_light_radiance_w_m2_sr_um
                if background_light_radiance_w_m2_sr_um is None
                else float(background_light_radiance_w_m2_sr_um)
            ),
            "weather": self.config.weather if weather is None else str(weather),
        }
        budget = self._budget(**condition)
        batch, symbol_count = symbols.shape
        receivers = self.config.num_receivers
        real_dtype = _real_dtype(symbols.dtype)
        fading_shape = (batch, receivers, 1)
        scintillation_power = self.sample_malaga_power(
            fading_shape,
            budget,
            symbols.device,
            real_dtype,
            generator,
        )

        attenuation_amplitude = 10.0 ** (-budget["total_attenuation_db"] / 20.0)
        # Table I reports transmitted optical power after the transmitter AMP.
        # Therefore its 30-dB gain must not be applied to the signal a second time.
        propagation_amplitude = attenuation_amplitude * scintillation_power.sqrt()

        time_index = torch.arange(
            symbol_count, device=symbols.device, dtype=real_dtype
        )
        phase = torch.zeros(
            (batch, receivers, symbol_count),
            device=symbols.device,
            dtype=real_dtype,
        )
        if self.config.include_doppler:
            phase = phase + (
                2.0
                * math.pi
                * budget["residual_doppler_hz"]
                / self.config.symbol_rate_baud
                * time_index
            ).view(1, 1, -1)
        if self.config.include_phase_noise and self.config.lo_linewidth_hz > 0:
            increment_std = math.sqrt(
                2.0
                * math.pi
                * self.config.lo_linewidth_hz
                / self.config.symbol_rate_baud
            )
            increments = torch.randn(
                phase.shape,
                device=symbols.device,
                dtype=real_dtype,
                generator=generator,
            ) * increment_std
            phase = phase + increments.cumsum(dim=-1)

        propagation_gain = propagation_amplitude.to(symbols.dtype) * torch.exp(
            1j * phase
        ).to(symbols.dtype)
        channel_gain = math.sqrt(budget["launch_power_w"]) * propagation_gain
        received = channel_gain * symbols[:, None, :]

        if self.config.include_background_noise and budget["background_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["background_noise_w"])
        if self.config.include_shot_noise and budget["receiver_shot_noise_w"] > 0:
            received = received + _complex_normal(
                received.shape, symbols.device, symbols.dtype, generator
            ) * math.sqrt(budget["receiver_shot_noise_w"])
        if self.config.include_ase_noise and budget["ase_noise_w"] > 0:
            transmitter_ase = _complex_normal(
                (batch, 1, symbol_count),
                symbols.device,
                symbols.dtype,
                generator,
            ) * math.sqrt(budget["ase_noise_w"])
            received = received + propagation_gain * transmitter_ase

        # The transmitter-EDFA ASE is one common source waveform, so diversity
        # cannot average it away. Receiver background noise is branch-independent.
        equivalent_noise_variance = torch.zeros(
            (batch, 1), device=symbols.device, dtype=real_dtype
        )
        if self.config.include_ase_noise:
            equivalent_noise_variance = equivalent_noise_variance + (
                budget["ase_noise_w"] / budget["launch_power_w"]
            )
        receiver_independent_noise_w = (
            budget["background_noise_w"] if self.config.include_background_noise else 0.0
        ) + (
            budget["receiver_shot_noise_w"] if self.config.include_shot_noise else 0.0
        )
        if receiver_independent_noise_w > 0:
            if receivers == 1 or self.config.combining.upper() == "MRC":
                combined_signal_power = channel_gain[..., :1].abs().square().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receiver_independent_noise_w
                    / combined_signal_power.clamp_min(torch.finfo(real_dtype).tiny)
                )
            else:
                combined_signal_amplitude = channel_gain[..., :1].abs().sum(dim=1)
                equivalent_noise_variance = equivalent_noise_variance + (
                    receivers
                    * receiver_independent_noise_w
                    / combined_signal_amplitude.square().clamp_min(
                        torch.finfo(real_dtype).tiny
                    )
                )
        if bool((equivalent_noise_variance > 0).any()):
            instantaneous_snr = equivalent_noise_variance.clamp_min(
                torch.finfo(real_dtype).tiny
            ).reciprocal()
            snr_linear_sum = float(instantaneous_snr.sum().item())
            snr_sample_count = int(instantaneous_snr.numel())
            mean_snr_linear = snr_linear_sum / max(snr_sample_count, 1)
            mean_snr_db = 10.0 * math.log10(max(mean_snr_linear, 1e-30))
        else:
            snr_linear_sum = float("inf")
            snr_sample_count = batch
            mean_snr_db = float("inf")
        diagnostics = {
            "channel_version": PAPER_FSO_CHANNEL_VERSION,
            "mean_snr_db": mean_snr_db,
            "snr_linear_sum": snr_linear_sum,
            "snr_sample_count": snr_sample_count,
            "num_receivers": receivers,
            "combining": self.config.combining.upper(),
            "budget": budget,
        }

        if receivers == 1:
            return FSOChannelOutput(
                symbols=received[:, 0, :],
                channel_gain=channel_gain[:, 0, :],
                is_equalized=False,
                diagnostics=diagnostics,
            )

        if self.config.combining.upper() == "MRC":
            denominator = (
                channel_gain.abs().square().sum(dim=1)
            ).clamp_min(torch.finfo(real_dtype).tiny)
            combined = (channel_gain.conj() * received).sum(dim=1) / denominator
        else:
            magnitudes = channel_gain.abs().clamp_min(1e-30)
            combined = (
                channel_gain.conj() / magnitudes * received
            ).sum(dim=1) / magnitudes.sum(dim=1).clamp_min(1e-30)
        return FSOChannelOutput(
            symbols=combined,
            is_equalized=True,
            diagnostics=diagnostics,
        )


@dataclass
class PaperFSOIndexChannel:
    receiver: object
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = BITS_PER_INDEX

    @property
    def receiver_config(self):
        return self.receiver.reproducibility_config()

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(
        self,
        indices,
        modulation="QPSK",
        return_statistics=False,
        **channel_kwargs,
    ):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported paper modulation: {modulation}")
        link = QAMIndexLink(
            self.receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=self.receiver_config,
        )
        return link(
            indices,
            return_statistics=return_statistics,
            **channel_kwargs,
        )


def paper_fig4_conditions():
    return [
        {
            "label": f"{modulation}, Pt={power} dBm, zenith={zenith} deg",
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": 20.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for power in (30, 33, 36)
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in range(0, 81, 10)
    ]


def paper_fig6_pointing_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt=36 dBm, zenith={zenith} deg, "
                f"pointing={pointing} urad"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": 36.0,
                "zenith_angle_deg": float(zenith),
                "pointing_error_urad": float(pointing),
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for zenith in (0, 30, 60)
        for pointing in range(18, 23)
    ]


def paper_fig6_background_conditions():
    return [
        {
            "label": (
                f"{modulation}, Pt={power} dBm, "
                f"BGL={radiance:g} W/m2/sr/um"
            ),
            "channel_kwargs": {
                "modulation": modulation,
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": float(radiance),
                "weather": "clear",
            },
        }
        for modulation in ("QPSK", "8QAM", "16QAM")
        for radiance in (1e-4, 1e-2, 1.0)
        for power in range(28, 37)
    ]


def paper_fig7_weather_conditions():
    return [
        {
            "label": f"QPSK, Pt={power} dBm, weather={weather}",
            "channel_kwargs": {
                "modulation": "QPSK",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 15.0,
                "scattered_power_ratio": 0.95,
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": weather,
            },
        }
        for weather in ("clear", "dust", "rain", "snow", "fog")
        for power in range(28, 37)
    ]


def paper_fig8_turbulence_conditions():
    """Return the common FSO-SC rho/Pt sweep, not the BPG codec pipelines."""
    return [
        {
            "label": f"16QAM, Pt={power} dBm, rho={rho}",
            "channel_kwargs": {
                "modulation": "16QAM",
                "launch_power_dbm": float(power),
                "zenith_angle_deg": 0.0,
                "pointing_error_urad": 19.0,
                "scattered_power_ratio": float(rho),
                "background_light_radiance_w_m2_sr_um": 1e-4,
                "weather": "clear",
            },
        }
        for rho in (0.95, 0.75, 0.596)
        for power in range(28, 37)
    ]


def paper_fig8_diversity_index_channels(base_config=None):
    """Expose the paper's 2/4-Rx EGC/MRC comparisons.

    The paper does not disclose inter-branch turbulence correlation, so the
    receiver implementation records its independent-block-fade assumption.
    """
    base_config = PaperFSOSystemConfig() if base_config is None else base_config
    channels = {
        "1Rx": PaperFSOIndexChannel(
            PaperMalagaFSOChannel(replace(base_config, num_receivers=1))
        )
    }
    for receivers in (2, 4):
        for combining in ("EGC", "MRC"):
            label = f"{receivers}Rx-{combining}"
            receiver_config = replace(
                base_config,
                num_receivers=receivers,
                combining=combining,
            )
            channels[label] = PaperFSOIndexChannel(
                PaperMalagaFSOChannel(receiver_config)
            )
    return channels


def paper_fso_equation_self_test():
    config = PaperFSOSystemConfig()
    reference = compute_paper_link_budget(
        config,
        launch_power_dbm=33.0,
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
        weather="clear",
    )
    assert abs(reference["slant_range_m"] - 620e3) < 1e-6
    assert abs(reference["geometric_scintillation_db"] - 28.69) < 0.05
    assert abs(reference["pointing_db"] - 35.68) < 0.05
    assert abs(reference["atmospheric_db"] - 2.94) < 1e-9
    # The paper omits beta and the coherent phase. With the documented pi/2
    # phase, its Eq. (15) has no feasible beta here; this must stay explicit.
    assert reference["malaga_beta_fit_feasible"] is False
    assert reference["malaga_beta_fit_relative_residual"] > 0.0
    try:
        compute_paper_link_budget(
            PaperFSOSystemConfig(beta_fit_policy="strict"),
            zenith_angle_deg=0.0,
            scattered_power_ratio=0.95,
        )
    except ValueError:
        pass
    else:
        raise AssertionError("Strict beta fitting must reject the underdetermined fit")
    expected_ranges_km = {
        0: 620.000,
        20: 655.958,
        30: 705.628,
        60: 1107.733,
        80: 1977.252,
    }
    for zenith, expected_km in expected_ranges_km.items():
        actual_km = satellite_slant_range_m(zenith, config) / 1000.0
        assert abs(actual_km - expected_km) < 0.001
    expected_weather_db = {
        "clear": 2.94,
        "dust": 20.86,
        "rain": 28.06,
        "snow": 42.90,
        "fog": 271.832,
    }
    for weather, expected_db in expected_weather_db.items():
        weather_budget = compute_paper_link_budget(config, weather=weather)
        assert abs(weather_budget["atmospheric_db"] - expected_db) < 0.001
    peak = compute_paper_link_budget(config, zenith_angle_deg=20.0)
    high = compute_paper_link_budget(config, zenith_angle_deg=80.0)
    assert abs(peak["doppler_hz"]) > abs(high["doppler_hz"])

    convergence_reference = compute_paper_link_budget(
        PaperFSOSystemConfig(integration_points=32769),
        zenith_angle_deg=0.0,
        pointing_error_urad=20.0,
    )
    assert abs(
        reference["rytov_variance"] / convergence_reference["rytov_variance"] - 1.0
    ) < 5e-3
    assert abs(
        reference["malaga_alpha"] / convergence_reference["malaga_alpha"] - 1.0
    ) < 5e-3

    receiver = PaperMalagaFSOChannel(config)
    generator_a = torch.Generator().manual_seed(123)
    generator_b = torch.Generator().manual_seed(123)
    sample_a = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_a
    )
    sample_b = receiver.sample_malaga_power(
        (4096, 1, 1), reference, torch.device("cpu"), torch.float32, generator_b
    )
    assert torch.equal(sample_a, sample_b)
    assert bool((sample_a >= 0).all()) and bool(torch.isfinite(sample_a).all())
    assert 0.9 < float(sample_a.mean()) < 1.1
    print(
        "Paper FSO Eq. (3)-(18) self-test passed: "
        f"asg={reference['geometric_scintillation_db']:.2f} dB, "
        f"ape={reference['pointing_db']:.2f} dB, "
        f"alpha={reference['malaga_alpha']:.2f}, beta={reference['malaga_beta']} "
        f"(Eq15 residual={reference['malaga_beta_fit_relative_residual']:.2%})"
    )


paper_fso_equation_self_test()


### Signal-estimated coherent receiver

진폭 block CSI는 사용하지만 실제 위상·송신 데이터로 carrier를 보정하지 않습니다. CFR/BPS 파라미터와 초기 phase ambiguity convention은 미공개 디테일을 보완한 선택입니다.


In [ ]:
"""Signal-estimated symbol-rate coherent carrier recovery for the FSO notebook.

Embed after ``qam_index_link`` and ``paper_fso_channel_v1``. This is a
plausible receiver implementation, NOT the undisclosed VPI DSP used by Chen
et al. It introduces no pilot symbols and never uses transmitted symbol values
or the simulated channel phase to recover the carrier. The default magnitude
mode retains explicitly assumed block-amplitude CSI; ``blind_rms`` is also
available, but its uniform-symbol assumption is unsuitable for strongly biased
semantic bitstreams.

Reference for feed-forward blind phase search: Pfau, Hoffmann, and Noe,
JLT 27(8), 989-999 (2009), doi:10.1109/JLT.2008.2010511. Fourth-power spectral
CFR is a conventional modulation-removal approximation, with its finite-frame
failure modes retained. Exact frame/timing/IQ calibration and an initial phase
branch near zero remain assumptions. No RRC/ADC waveform is synthesized here.
"""

from dataclasses import asdict, dataclass
import math

import torch
import torch.nn.functional as F


SIGNAL_DSP_VERSION = "fourth_power_fft_newton_bps_v1"
SIGNAL_DSP_REFERENCE = {
    "blind_phase_search": "https://doi.org/10.1109/JLT.2008.2010511",
    "scope": "symbol-rate, single receiver; no waveform/timing/IQ calibration model",
    "cfr": "fourth-power FFT peak plus continuous periodogram Newton refinement",
    "cpr": "windowed minimum-distance blind phase search and symmetry unwrapping",
    "phase_ambiguity": (
        "initial carrier phase lies in the branch nearest zero, consistent with "
        "the simulator phase origin; no transmitted data is used to select a branch"
    ),
    "no_pilots": True,
    "amplitude_default": "known block magnitude only, not the complex channel gain",
    "failure_policy": "keep estimates/errors; do not drop low-SNR frames",
}


@dataclass(frozen=True)
class SignalDSPConfig:
    amplitude_mode: str = "known_block_magnitude"
    cfo_fft_oversampling: int = 4
    cfo_newton_steps: int = 4
    bps_test_phases: int = 64
    bps_window_symbols: int = 65
    cfo_enabled: bool = True
    cpr_enabled: bool = True

    def __post_init__(self):
        if self.amplitude_mode not in {"known_block_magnitude", "blind_rms"}:
            raise ValueError("Unsupported amplitude_mode")
        if self.cfo_fft_oversampling < 1 or self.cfo_newton_steps < 0:
            raise ValueError("Invalid CFR FFT/refinement configuration")
        if self.bps_test_phases < 8:
            raise ValueError("Use at least eight BPS trial phases")
        if self.bps_window_symbols < 1 or self.bps_window_symbols % 2 != 1:
            raise ValueError("BPS window must be a positive odd integer")


def _qam_axis_levels(modulation, device, dtype):
    modulation = modulation.upper()
    if modulation == "QPSK":
        i = q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(2)
        period = math.pi / 2
    elif modulation == "16QAM":
        i = q = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(10)
        period = math.pi / 2
    elif modulation == "8QAM":
        # Rectangular geometry in qam_constellation, NOT square-QAM symmetry.
        i = torch.tensor([-3.0, -1.0, 1.0, 3.0], device=device, dtype=dtype) / math.sqrt(6)
        q = torch.tensor([-1.0, 1.0], device=device, dtype=dtype) / math.sqrt(6)
        period = math.pi
    else:
        raise ValueError(f"Unsupported modulation: {modulation}")
    return i, q, period


def _estimate_cfo_fourth_power(observed, symbol_rate_baud, config):
    batch, count = observed.shape
    # The fourth harmonic repeats at Fs/4, so the unambiguous CFO interval is
    # [-Fs/8, Fs/8). Paper residual offsets are within that acquisition range.
    nfft = 1 << max(1, (count * config.cfo_fft_oversampling - 1).bit_length())
    # Severe fog can make gain-normalized receiver noise enormous. Remove a
    # positive block scale before the fourth power; it does not change the
    # frequency peak and prevents overflow in frames whose data is unrecoverable.
    peak_scale = observed.abs().amax(dim=-1, keepdim=True).clamp_min(
        torch.finfo(observed.real.dtype).tiny
    )
    cfr_samples = observed / peak_scale
    fourth = cfr_samples.square().square()
    spectrum = torch.fft.fft(fourth, n=nfft, dim=-1)
    power = spectrum.abs().square()
    peak = power.argmax(dim=-1)
    signed_peak = torch.where(peak > nfft // 2, peak - nfft, peak)
    real_dtype = observed.real.dtype
    omega = signed_peak.to(real_dtype) * (2.0 * math.pi / nfft)
    index = torch.arange(count, device=observed.device, dtype=real_dtype)
    centered = index - (count - 1) / 2.0
    max_step = 2.0 * math.pi / nfft
    for _ in range(config.cfo_newton_steps):
        derotated = fourth * torch.exp(-1j * omega[:, None] * centered[None, :])
        value = derotated.sum(dim=-1)
        derivative = (-1j * centered[None, :] * derotated).sum(dim=-1)
        curvature = (-centered.square()[None, :] * derotated).sum(dim=-1)
        slope = 2 * (derivative * value.conj()).real
        second = 2 * ((curvature * value.conj()).real + derivative.abs().square())
        stable = second < -torch.finfo(real_dtype).eps
        denominator = torch.where(stable, second, torch.ones_like(second))
        step = torch.where(stable, slope / denominator, torch.zeros_like(slope))
        omega = omega - step.clamp(min=-max_step, max=max_step)
    omega = torch.remainder(omega + math.pi, 2 * math.pi) - math.pi
    estimated_hz = omega * symbol_rate_baud / (8 * math.pi)
    phase = omega[:, None] * index[None, :] / 4.0
    compensated = observed * torch.exp(-1j * phase)
    peak_ratio = power.max(dim=-1).values / power.mean(dim=-1).clamp_min(
        torch.finfo(real_dtype).tiny
    )
    return compensated, estimated_hz, peak_ratio


def _unwrap_symmetry_phase(phase, period):
    differences = torch.diff(phase, dim=-1)
    wrapped = torch.remainder(differences + period / 2, period) - period / 2
    corrections = wrapped - differences
    cumulative = torch.cat(
        [torch.zeros_like(phase[:, :1]), corrections.cumsum(dim=-1)], dim=-1
    )
    return phase + cumulative, (corrections.abs() > period / 2).sum(dim=-1)


def _blind_phase_search(observed, modulation, config):
    i_levels, q_levels, period = _qam_axis_levels(
        modulation, observed.device, observed.real.dtype
    )
    trial_phases = (
        torch.arange(config.bps_test_phases, device=observed.device, dtype=observed.real.dtype)
        / config.bps_test_phases - 0.5
    ) * period
    rotated = observed[:, None, :] * torch.exp(-1j * trial_phases[None, :, None])
    i_distance = (rotated.real[..., None] - i_levels).square().amin(dim=-1)
    q_distance = (rotated.imag[..., None] - q_levels).square().amin(dim=-1)
    costs = F.avg_pool1d(
        i_distance + q_distance,
        kernel_size=config.bps_window_symbols,
        stride=1,
        padding=config.bps_window_symbols // 2,
        count_include_pad=False,
    )
    selected = trial_phases[costs.argmin(dim=1)]
    phase, unwrap_events = _unwrap_symmetry_phase(selected, period)
    # Global rotational ambiguity is not solved using the source symbols. The
    # initial branch near zero is an explicit simulator convention, not a pilot.
    phase = phase - torch.round(phase[:, :1] / period) * period
    corrected = observed * torch.exp(-1j * phase)
    return corrected, phase, unwrap_events


@torch.no_grad()
def recover_qam_carrier(
    received,
    modulation,
    symbol_rate_baud,
    config=None,
    known_block_magnitude=None,
):
    """Estimate CFR/CPR from received samples, without source data/true phase.

    ``known_block_magnitude`` may contain positive block-amplitude CSI only.
    It must not contain the true phase trajectory or the transmitted symbols.
    """
    config = SignalDSPConfig() if config is None else config
    if received.ndim != 2 or not torch.is_complex(received) or received.shape[-1] < 8:
        raise ValueError("Expected complex [batch, symbols>=8] receiver samples")
    if symbol_rate_baud <= 0 or not torch.isfinite(received).all():
        raise ValueError("Invalid receiver samples or symbol rate")
    if config.amplitude_mode == "known_block_magnitude":
        if known_block_magnitude is None or torch.is_complex(known_block_magnitude):
            raise ValueError("Positive REAL block-amplitude CSI is required")
        if not torch.isfinite(known_block_magnitude).all() or (known_block_magnitude <= 0).any():
            raise ValueError("Invalid block-amplitude CSI")
        amplitude = torch.broadcast_to(known_block_magnitude, (received.shape[0], 1))
    else:
        amplitude = received.abs().square().mean(dim=-1, keepdim=True).sqrt()
        amplitude = amplitude.clamp_min(torch.finfo(received.real.dtype).tiny)
    observed = received / amplitude
    zeros = torch.zeros(received.shape[0], device=received.device, dtype=received.real.dtype)
    if config.cfo_enabled:
        observed, cfo_hz, peak_ratio = _estimate_cfo_fourth_power(
            observed, symbol_rate_baud, config
        )
    else:
        cfo_hz, peak_ratio = zeros, zeros
    if config.cpr_enabled:
        observed, phase, unwrap_events = _blind_phase_search(observed, modulation, config)
    else:
        phase = torch.zeros_like(received.real)
        unwrap_events = zeros.to(torch.int64)
    diagnostics = {
        "receiver_version": SIGNAL_DSP_VERSION,
        "amplitude_mode": config.amplitude_mode,
        "estimated_cfo_hz": cfo_hz.detach().cpu().tolist(),
        "cfr_peak_to_mean_power": peak_ratio.detach().cpu().tolist(),
        "cfr_unambiguous_interval_hz": [-symbol_rate_baud / 8, symbol_rate_baud / 8],
        "estimated_phase_start_rad": phase[:, 0].detach().cpu().tolist(),
        "estimated_phase_end_rad": phase[:, -1].detach().cpu().tolist(),
        "phase_symmetry_unwrap_events": unwrap_events.detach().cpu().tolist(),
        "failure_policy": "estimates/errors retained, no frame rejection",
    }
    return observed, diagnostics


@dataclass
class SignalDSPFSOReceiver:
    """Replace phase-oracle equalization for one physical receiver branch."""

    physical_receiver: object
    modulation: str = "QPSK"
    dsp_config: object = None

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.physical_receiver.config.num_receivers != 1:
            raise ValueError("Signal DSP wrapper supports single Rx only; no oracle MRC")

    def reproducibility_config(self):
        return {
            "physical_receiver": self.physical_receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
            "modulation": self.modulation.upper(),
        }

    @torch.no_grad()
    def __call__(self, symbols, **channel_kwargs):
        observation = self.physical_receiver(symbols, **channel_kwargs)
        if observation.is_equalized:
            raise ValueError("Expected un-equalized physical receiver observation")
        magnitude = None
        if self.dsp_config.amplitude_mode == "known_block_magnitude":
            # Deliberately read only real block magnitude, NOT its phase. Do
            # not pass complex channel_gain to recover_qam_carrier.
            magnitude = observation.channel_gain.abs().mean(dim=-1, keepdim=True)
        recovered, receiver_diagnostics = recover_qam_carrier(
            observation.symbols,
            self.modulation,
            self.physical_receiver.config.symbol_rate_baud,
            self.dsp_config,
            known_block_magnitude=magnitude,
        )
        diagnostics = dict(observation.diagnostics or {})
        diagnostics["signal_dsp"] = receiver_diagnostics
        return FSOChannelOutput(symbols=recovered, is_equalized=True, diagnostics=diagnostics)


@dataclass
class SignalDSPPaperIndexChannel:
    """Drop-in single-Rx alternative to PaperFSOIndexChannel in this notebook."""

    receiver: object
    dsp_config: object = None
    modulations: tuple = ("QPSK", "8QAM", "16QAM")
    bits_per_index: int = 14

    def __post_init__(self):
        if self.dsp_config is None:
            self.dsp_config = SignalDSPConfig()
        if self.receiver.config.num_receivers != 1:
            raise ValueError("Estimated carrier receiver is currently single-Rx only")

    @property
    def receiver_config(self):
        return {
            "physical_receiver": self.receiver.reproducibility_config(),
            "receiver_version": SIGNAL_DSP_VERSION,
            "receiver_config": asdict(self.dsp_config),
            "receiver_assumptions": SIGNAL_DSP_REFERENCE,
        }

    def reproducibility_config(self):
        return {
            "qam_mapping_version": QAM_MAPPING_VERSION,
            "modulations": tuple(name.upper() for name in self.modulations),
            "bits_per_index": self.bits_per_index,
            "receiver": self.receiver_config,
        }

    def __call__(self, indices, modulation="QPSK", return_statistics=False, **channel_kwargs):
        modulation = modulation.upper()
        if modulation not in tuple(name.upper() for name in self.modulations):
            raise ValueError(f"Unsupported modulation: {modulation}")
        receiver = SignalDSPFSOReceiver(self.receiver, modulation, self.dsp_config)
        link = QAMIndexLink(
            receiver,
            modulation=modulation,
            bits_per_index=self.bits_per_index,
            receiver_config=receiver.reproducibility_config(),
        )
        return link(indices, return_statistics=return_statistics, **channel_kwargs)


def signal_dsp_offline_self_test(device="cpu"):
    """Synthetic local checks only; does not download weights or run training."""
    device = torch.device(device)
    generator = torch.Generator(device=device).manual_seed(71819)
    count = 4096
    sample_rate = 28e9
    time = torch.arange(count, device=device) / sample_rate
    records = {}
    for modulation in ("QPSK", "8QAM", "16QAM"):
        points, _, _ = qam_constellation(modulation, device)
        ids = torch.randint(len(points), (2, count), generator=generator, device=device)
        source = points[ids]
        true_cfo = torch.tensor([700e6, -320e6], device=device)
        phase = 0.03 + 2 * math.pi * true_cfo[:, None] * time[None, :]
        amplitude = torch.tensor([[0.002], [0.0005]], device=device)
        noise = torch.complex(
            torch.randn(source.shape, generator=generator, device=device),
            torch.randn(source.shape, generator=generator, device=device),
        ) * (0.005 / math.sqrt(2))
        received = amplitude * (source * torch.exp(1j * phase) + noise)
        recovered, stats = recover_qam_carrier(
            received, modulation, sample_rate, known_block_magnitude=amplitude
        )
        recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
        errors = int((recovered_ids != ids).sum())
        estimated_cfo = torch.tensor(stats["estimated_cfo_hz"], device=device)
        assert errors == 0, (modulation, errors, stats)
        # Finite-frame fourth-moment self-noise is nonzero for QAM. Require
        # error below one 4th-harmonic FFT bin; BPS must still give exact data.
        assert (estimated_cfo - true_cfo).abs().max() < sample_rate / (4 * count), stats
        records[modulation] = {"symbol_errors": errors, "estimated_cfo_hz": stats["estimated_cfo_hz"]}

    # Nonzero phase trajectory must be inferred and unwrapped, not divided out
    # with a hidden simulator phase. Deliberately turn CFR off for this check.
    points, _, _ = qam_constellation("QPSK", device)
    ids = torch.randint(len(points), (1, count), generator=generator, device=device)
    source = points[ids]
    phase = torch.linspace(0, 3, count, device=device)[None, :]
    recovered, stats = recover_qam_carrier(
        source * torch.exp(1j * phase), "QPSK", sample_rate,
        SignalDSPConfig(cfo_enabled=False), torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert torch.equal(ids, recovered_ids), stats
    assert stats["phase_symmetry_unwrap_events"][0] >= 1

    # A global pi/2 ambiguity is intentionally NOT resolved using source data.
    received = source * torch.exp(torch.tensor(1j * math.pi / 2, device=device))
    recovered, _ = recover_qam_carrier(
        received, "QPSK", sample_rate,
        known_block_magnitude=torch.ones((1, 1), device=device),
    )
    recovered_ids = (recovered[..., None] - points).abs().square().argmin(dim=-1)
    assert not torch.equal(ids, recovered_ids), "Undisclosed source-aided phase ambiguity resolution"
    print("Signal-estimated CFR/CPR offline self-test passed:", records)
    return records


### VGG perceptual loss / LPIPS

In [ ]:
!pip -q install --no-deps "lpips==0.1.4"

from importlib.metadata import version
import platform
import lpips
import torch
import torch.nn as nn
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

vgg_loss = lpips.LPIPS(net="vgg").to(device).eval()
vgg_loss.requires_grad_(False)
mse_loss = nn.MSELoss()
LPIPS_PACKAGE_VERSION = version("lpips")
RUNTIME_INFO = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "lpips": LPIPS_PACKAGE_VERSION,
}
PAPER_RUNTIME = {
    "torch": "2.0.0",
    "cuda": "11.8",
    "gpu": "A100",
}
TORCHVISION_COMPATIBILITY_ASSUMPTION = "0.15.1"
STRICT_PAPER_RUNTIME = False
print("Runtime:", RUNTIME_INFO)
runtime_matches_paper = (
    torch.__version__.startswith(PAPER_RUNTIME["torch"])
    and str(torch.version.cuda) == PAPER_RUNTIME["cuda"]
    and RUNTIME_INFO["gpu"] is not None
    and PAPER_RUNTIME["gpu"] in RUNTIME_INFO["gpu"]
)
if not runtime_matches_paper:
    print(
        "Warning: the paper reports A100/PyTorch 2.0.0/CUDA 11.8. "
        "This notebook records, but does not silently replace, the active runtime."
    )
    if STRICT_PAPER_RUNTIME:
        raise RuntimeError(
            f"Strict paper runtime required {PAPER_RUNTIME}, got {RUNTIME_INFO}"
        )

## 2. 학습 및 중단·재개 설정

최근 committed optimizer update부터 재개하며 그 이후 미저장 update는 다시 수행합니다. epoch 예산을 늘려도 같은 checkpoint에서 이어갑니다. 데이터나 학습 설정을 바꾸면 새 RUN_NAME을 사용하세요.


In [ ]:
"""Paper-objective training and recoverable checkpoints; no work at import."""
import copy
from contextlib import contextmanager
import hashlib
import json
import math
import os
from pathlib import Path
import random
import shutil
import threading
import time
import uuid

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset


TRAINING_ENGINE_VERSION = "paper_faithful_v6"
EXECUTION_REVISION = "A100_batch4_latest_best_io_v3"

# These affect execution only, not the data, objective, effective batch or optimizer.
OPERATIONAL_KEYS = frozenset({"micro_batch_size", "num_workers", "log_every_updates",
                              "checkpoint_every_updates", "activation_checkpointing"})


def learning_configuration(configuration):
    return {key: value for key, value in configuration.items() if key not in OPERATIONAL_KEYS}


def require_learning_compatibility(saved, current):
    if learning_configuration(saved) != learning_configuration(current):
        raise RuntimeError("Checkpoint model/data/objective/effective-batch configuration differs; "
                           "performance migration may not change learning settings")
    return {key: {"from": saved.get(key), "to": current.get(key)}
            for key in sorted(OPERATIONAL_KEYS) if saved.get(key) != current.get(key)}


def capture_rng():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] is not None and torch.cuda.is_available():
        if len(state["cuda"]) != torch.cuda.device_count():
            raise RuntimeError("CUDA device count changed; exact RNG continuation is unavailable")
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda"]])


def sha256_path(path, progress=None):
    digest = hashlib.sha256()
    total = Path(path).stat().st_size if progress is not None else 0
    completed = 0
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
            completed += len(block)
            if progress is not None:
                progress(completed, total)
    return digest.hexdigest()


def copy_checkpoint_with_progress(source, destination, progress=None):
    """Copy only the checkpoint being written; never inspect an older Drive file."""
    total = Path(source).stat().st_size
    completed = 0
    with Path(source).open("rb") as reader, Path(destination).open("wb") as writer:
        for block in iter(lambda: reader.read(8 * 1024**2), b""):
            writer.write(block)
            completed += len(block)
            if progress is not None:
                progress(completed, total)


def trusted_torch_load(path):
    # Use only self-created run checkpoints, never untrusted external pickle files.
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")






class BitAwareControlledIndexErrors:
    """Exact corrupted-token count; conditional independent-bit XOR masks.

    q describes a BSC-inspired surrogate, not measured physical-channel BER.
    Error count is round(T * (1-(1-q)**bits)); error positions are sampled uniformly.
    """

    def __init__(self, bits=14, reference_ber=(0.0, 1e-4, 1e-3, 5e-3, 1e-2)):
        self.bits = int(bits)
        self.reference_ber = tuple(float(q) for q in reference_ber)
        if self.bits < 1 or any(not 0 <= q < .5 for q in self.reference_ber):
            raise ValueError("Invalid bit-error surrogate configuration")

    @torch.no_grad()
    def __call__(self, indices, reference_ber=None, generator=None):
        q = (random.choice(self.reference_ber) if reference_ber is None
             else float(reference_ber))
        if not 0 <= q < .5:
            raise ValueError("reference_ber must be in [0,.5)")
        if indices.min() < 0 or indices.max() >= 2**self.bits:
            raise ValueError("Index outside bit width")
        flat = indices.reshape(indices.shape[0], -1)
        count = round(flat.shape[1] * (1 - (1 - q)**self.bits))
        output = flat.clone()
        if count:
            k = torch.arange(1, self.bits+1, dtype=torch.float64, device=indices.device)
            combinations = torch.tensor([math.comb(self.bits, j) for j in range(1, self.bits+1)],
                                        dtype=torch.float64, device=indices.device)
            probability = combinations * q**k * (1-q)**(self.bits-k)
            probability /= probability.sum()
            binary_weights = (1 << torch.arange(self.bits, device=indices.device, dtype=torch.long))
            for b in range(flat.shape[0]):
                positions = torch.randperm(flat.shape[1], device=indices.device,
                                          generator=generator)[:count]
                flips = torch.multinomial(probability, count, replacement=True,
                                          generator=generator) + 1
                ranks = torch.rand((count, self.bits), device=indices.device,
                                   generator=generator).argsort(dim=1).argsort(dim=1)
                masks = ((ranks < flips[:, None]).long() * binary_weights[None]).sum(dim=1)
                output[b, positions] = torch.bitwise_xor(flat[b, positions], masks)
        actual = (output != flat).sum(dim=1)
        assert bool((actual == count).all())
        return output.reshape_as(indices), {"reference_ber": q, "errors_per_image": count}


class PerImageBitAwareControlledIndexErrors(BitAwareControlledIndexErrors):
    """Match legacy microbatch-1 q and bit-mask draws independently of batching."""

    @torch.no_grad()
    def __call__(self, indices, reference_ber=None, generator=None):
        received, strengths, counts = [], [], []
        for image in indices.split(1, dim=0):
            output, info = super().__call__(image, reference_ber, generator)
            received.append(output)
            strengths.append(info["reference_ber"])
            counts.append(info["errors_per_image"])
        return torch.cat(received, dim=0), {
            "reference_ber": sum(strengths)/len(strengths),
            "errors_per_image": sum(counts)/len(counts),
            "per_image_reference_ber": strengths,
            "per_image_errors": counts,
        }


def faithful_forward_loss(model, images, perceptual_loss, stage, error_model=None):
    quantized, vq_loss, indices = model.encode(images)
    reference_ber = 0.0
    token_errors = 0
    if stage == 2:
        received, info = error_model(indices)
        received_vectors = model.vq.lookup(received)
        # Reconstruction gradients use STE; Eq28 uses the original clean assignments.
        quantized = quantized + (received_vectors.to(quantized.dtype)-quantized).detach()
        reference_ber = info["reference_ber"]
        token_errors = info["errors_per_image"]
    reconstruction = model.decode_quantized(quantized)
    mse = F.mse_loss(reconstruction, images)
    perceptual = perceptual_loss(reconstruction, images).mean()
    total = mse + .1 * perceptual + vq_loss
    metrics = {"loss": float(total.detach()), "mse": float(mse.detach()),
               "lpips": float(perceptual.detach()), "vq": float(vq_loss.detach()),
               "injected_token_errors": float(token_errors), "reference_ber": reference_ber}
    return total, metrics, indices.detach()


def train_paper_stage(model, dataset, perceptual_loss, device, stage, epochs,
                      configuration, store, loss_function=None, stop_after_updates=None,
                      require_resume=False, best_metric="lpips"):
    """Resume at optimizer boundaries. Deterministic preprocessing is required.

    Epoch budget may be extended on resume; changing any learning/data option is rejected.
    stop_after_updates is a test/pause budget, never part of the paper protocol.
    Stores that implement save_best retain the lowest completed-epoch training
    mean of best_metric. This does not inspect validation/test data or perform an
    additional forward pass; the last-epoch model remains the training outcome.
    """
    if stage not in (1, 2):
        raise ValueError("stage must be 1 or 2")
    effective = int(configuration["effective_batch_size"])
    micro = int(configuration["micro_batch_size"])
    if not 1 <= micro <= effective or effective % micro:
        raise ValueError("effective batch must be divisible by positive micro batch")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(.5, .9))
    error_model = PerImageBitAwareControlledIndexErrors(reference_ber=configuration["stage2_reference_ber"])
    state = store.load()
    if state is None:
        if require_resume:
            raise RuntimeError("Final weights exist but resumable Adam/progress checkpoint is missing; "
                               "cannot extend this run without changing its optimization history")
        progress = {"epoch": 0, "cursor": 0, "order": None, "updates": 0,
                    "metric_sums": {}, "epoch_samples": 0,
                    "code_histogram": None, "history": []}
    else:
        if state["engine_version"] != TRAINING_ENGINE_VERSION or state["stage"] != stage:
            raise RuntimeError("Checkpoint model/data/training configuration differs; use a new RUN_NAME")
        changes = require_learning_compatibility(state["configuration"], configuration)
        if changes:
            print("Validated execution-only changes:", changes, flush=True)
        model.load_state_dict(state["state_dict"], strict=True)
        optimizer.load_state_dict(state["optimizer"])
        progress = state["progress"]
        restore_rng(state["rng"])
        del state
    optimizer.zero_grad(set_to_none=True)
    model.train()

    def checkpoint():
        # No accumulated gradients remain at this boundary.
        store.save({"engine_version": TRAINING_ENGINE_VERSION, "stage": stage,
                    "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "configuration": configuration, "progress": progress, "rng": capture_rng(),
                    "execution_revision": EXECUTION_REVISION})

    def save_completed_epoch_best():
        save_best = getattr(store, "save_best", None)
        if save_best is None:
            return  # Older stores preserve their original behavior.
        # Only the actual weights at a finalized epoch boundary are eligible.
        # A historical minimum alone cannot recover that epoch's model weights.
        if (progress["order"] is not None or progress["cursor"] != 0
                or not progress["history"]):
            return
        completed = progress["history"][-1]
        if completed.get("epoch") != progress["epoch"]:
            raise RuntimeError("Finalized epoch history does not match current model state")
        if best_metric not in completed:
            raise KeyError(f"Best-model training metric is unavailable: {best_metric}")
        score = float(completed[best_metric])
        if not math.isfinite(score):
            raise FloatingPointError("Nonfinite best-model training metric")
        save_best(model.state_dict(), progress, score, best_metric, metadata={
            "engine_version": TRAINING_ENGINE_VERSION, "stage": stage,
            "configuration": configuration, "execution_revision": EXECUTION_REVISION,
            "selection_source": "completed_epoch_training_mean",
        })

    # If a runtime stopped after the finalized full checkpoint but before the
    # smaller best artifact committed, retry from those exact saved weights.
    save_completed_epoch_best()

    while progress["epoch"] < epochs:
        if progress["order"] is None:
            progress["order"] = torch.randperm(len(dataset)).tolist()
            progress["cursor"] = 0
            progress["metric_sums"] = {}
            progress["epoch_samples"] = 0
            progress["code_histogram"] = None
        order = progress["order"]
        if sorted(order) != list(range(len(dataset))) or not 0 <= progress["cursor"] <= len(dataset):
            raise RuntimeError("Invalid checkpoint data order/cursor")
        if progress["cursor"] != len(dataset) and progress["cursor"] % effective:
            raise RuntimeError("Checkpoint cursor is not an optimizer boundary")
        remaining = Subset(dataset, order[progress["cursor"]:])
        # Worker/iterator initialization must not consume the model/noise RNG stream.
        loader_rng = torch.Generator().manual_seed(0)
        loader = DataLoader(remaining, batch_size=micro, shuffle=False,
                            num_workers=int(configuration["num_workers"]),
                            pin_memory=torch.device(device).type == "cuda",
                            generator=loader_rng, persistent_workers=False)
        iterator = iter(loader)
        started = time.monotonic()
        while progress["cursor"] < len(dataset):
            group_size = min(effective, len(dataset) - progress["cursor"])
            consumed = 0
            while consumed < group_size:
                batch = next(iterator)
                if isinstance(batch, (tuple, list)):
                    batch = batch[0]
                batch = batch.to(device, non_blocking=True)
                count = batch.shape[0]
                if consumed + count > group_size:
                    raise RuntimeError("Microbatch crossed an optimizer-group boundary")
                if loss_function is None:
                    loss, metrics, indices = faithful_forward_loss(
                        model, batch, perceptual_loss, stage, error_model)
                else:
                    loss, metrics, indices = loss_function(model, batch, stage, error_model)
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError("Nonfinite loss; previous committed checkpoint preserved")
                (loss * (count/group_size)).backward()
                for key, value in metrics.items():
                    progress["metric_sums"][key] = progress["metric_sums"].get(key, 0.0) + value*count
                if indices is not None:
                    bins = torch.bincount(indices.flatten().cpu(), minlength=int(configuration["codebook_size"]))
                    if progress["code_histogram"] is None:
                        progress["code_histogram"] = bins
                    else:
                        progress["code_histogram"] += bins
                consumed += count
                progress["cursor"] += count
                progress["epoch_samples"] += count
                del loss, batch
            clip = configuration["gradient_clip_value"]
            if clip is not None:
                torch.nn.utils.clip_grad_value_(model.parameters(), float(clip))
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            progress["updates"] += 1
            if progress["updates"] % int(configuration["log_every_updates"]) == 0:
                average = progress["metric_sums"]["loss"] / max(progress["epoch_samples"], 1)
                print(f"Stage{stage} epoch {progress['epoch']+1}/{epochs} | "
                      f"{progress['cursor']}/{len(dataset)} images | update {progress['updates']} | "
                      f"loss {average:.5f} | epoch elapsed {time.monotonic()-started:.0f}s | "
                      f"micro {micro} | GPU peak "
                      f"{torch.cuda.max_memory_allocated()/1024**3 if torch.device(device).type == 'cuda' else 0:.1f} GiB", flush=True)
            epoch_finished = progress["cursor"] == len(dataset)
            stop_requested = (stop_after_updates is not None
                              and progress["updates"] >= stop_after_updates)
            periodic_save = progress["updates"] % int(configuration["checkpoint_every_updates"]) == 0
            # The finalized epoch checkpoint below supersedes a periodic or pause
            # checkpoint at this same update. A mid-epoch pause likewise needs
            # only one save even when its periodic timer also fires.
            if not epoch_finished and (periodic_save or stop_requested):
                checkpoint()
            if stop_requested and not epoch_finished:
                return progress
        means = {key: value/max(progress["epoch_samples"], 1)
                 for key, value in progress["metric_sums"].items()}
        histogram = progress["code_histogram"]
        if histogram is not None:
            probability = histogram.float() / histogram.sum().clamp_min(1)
            positive = probability > 0
            means["codes_used"] = int(positive.sum())
            means["code_perplexity"] = float(torch.exp(-(probability[positive]*probability[positive].log()).sum()))
            if means["codes_used"] < 16:
                print("Warning: fewer than 16 codes used this epoch; inspect training before trusting results.", flush=True)
        progress["history"].append({"epoch": progress["epoch"]+1, **means})
        print(f"Stage{stage} epoch {progress['epoch']+1}: {means}", flush=True)
        progress["epoch"] += 1
        progress["order"] = None
        progress["cursor"] = 0
        checkpoint()
        save_completed_epoch_best()
        if stop_after_updates is not None and progress["updates"] >= stop_after_updates:
            return progress
    return progress




"""No-update fit checks and training-only reconstruction diagnostics."""
import json
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageOps
import torch


def choose_microbatch_no_update(model, dataset, perceptual_loss, device,
                                requested=4, candidates=(4, 2, 1), reserve_gib=12):
    """Measure forward/backward fit without optimizer.step or persistent RNG changes.

    Adam moments are not yet on GPU. Reserve twelve GiB for them/workspaces.
    Only current stateless GroupNorm/dropout-zero models are eligible for this probe.
    """
    device = torch.device(device)
    if device.type != "cuda":
        raise RuntimeError("GPU memory-fit probe requires CUDA")
    if any(parameter.grad is not None for parameter in model.parameters()):
        raise RuntimeError("Probe must run before training, with no accumulated gradients")
    if any(isinstance(module, torch.nn.modules.batchnorm._BatchNorm)
           for module in model.modules()):
        raise RuntimeError("Probe cannot silently modify BatchNorm running state")
    rng = capture_rng()
    training = model.training
    model.train()
    observations = []
    try:
        for micro in [value for value in candidates if value <= requested]:
            batch = loss = images = reconstruction = quantized = indices = None
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
            started = time.perf_counter()
            try:
                images = torch.stack([dataset[index] for index in range(micro)]).to(device)
                quantized, vq, indices = model.encode(images)
                reconstruction = model.decode_quantized(quantized)
                loss = torch.nn.functional.mse_loss(reconstruction, images) + .1*perceptual_loss(
                    reconstruction, images).mean() + vq
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError("Nonfinite loss during no-update memory probe")
                loss.backward()
                torch.cuda.synchronize(device)
                allocated = torch.cuda.max_memory_allocated(device) / 1024**3
                reserved = torch.cuda.max_memory_reserved(device) / 1024**3
                total = torch.cuda.get_device_properties(device).total_memory / 1024**3
                okay = reserved + reserve_gib <= total
                record = {"micro_batch_size": micro, "peak_allocated_gib": allocated,
                          "peak_reserved_gib": reserved, "total_gib": total,
                          "reserved_for_Adam_workspaces_gib": reserve_gib,
                          "forward_backward_seconds": time.perf_counter()-started,
                          "fits_with_reserve": okay, "optimizer_updates": 0}
                observations.append(record)
                print("No-update memory probe:", record, flush=True)
                if okay:
                    return micro, observations
            except torch.cuda.OutOfMemoryError:
                observations.append({"micro_batch_size": micro, "oom": True,
                                     "optimizer_updates": 0})
                print(f"micro={micro} exceeded available GPU memory; trying smaller batch", flush=True)
            finally:
                model.zero_grad(set_to_none=True)
                batch = loss = images = reconstruction = quantized = indices = None
                if "vq" in locals():
                    del vq
                torch.cuda.empty_cache()
        raise RuntimeError("No safe no-recompute microbatch fitted. Do not change the objective; "
                           "continue with the original checkpointed notebook or inspect GPU allocations.")
    finally:
        model.zero_grad(set_to_none=True)
        model.train(training)
        restore_rng(rng)


@torch.inference_mode()
def training_snapshot(model, dataset, perceptual_loss, device, selected_indices, output_dir,
                      label, codebook_size=16384):
    """One stationary snapshot from TRAIN images, never test/model-selection data."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if not label.replace("_", "").isalnum():
        raise ValueError("Use a simple snapshot label")
    rng = capture_rng()
    training = model.training
    model.eval()
    histogram = torch.zeros(codebook_size, dtype=torch.long)
    records = []
    sheet = Image.new("RGB", (2*256, len(selected_indices)*(256+24)), "white")
    draw = ImageDraw.Draw(sheet)
    try:
        for row, index in enumerate(selected_indices):
            images = dataset[index].unsqueeze(0).to(device)
            reconstruction, vq_loss, indices = model(images)
            if not bool(torch.isfinite(reconstruction).all()):
                raise FloatingPointError("Nonfinite stationary reconstruction")
            clipped = reconstruction.clamp(-1, 1)
            mse = float(torch.nn.functional.mse_loss(reconstruction, images))
            perceptual = float(perceptual_loss(reconstruction, images).mean())
            histogram += torch.bincount(indices.flatten().cpu(), minlength=codebook_size)
            records.append({"training_index": index, "raw_mse": mse,
                            "raw_vgg_lpips": perceptual, "vq": float(vq_loss)})
            for column, tensor in enumerate((images, clipped)):
                array = ((tensor[0].permute(1, 2, 0).cpu().numpy()+1)*127.5).clip(0, 255).astype(np.uint8)
                sheet.paste(Image.fromarray(array), (column*256, row*280+24))
            draw.text((4, row*280+5), f"TRAIN {index} | input / reconstruction | MSE {mse:.4f}", fill="black")
        positive = histogram > 0
        probability = histogram[positive].double()/histogram.sum()
        report = {"label": label, "source": "fixed training images only; not evaluation/test",
                  "stationary_model": True, "images": len(records),
                  "mse_mean": sum(record["raw_mse"] for record in records)/len(records),
                  "vgg_lpips_mean": sum(record["raw_vgg_lpips"] for record in records)/len(records),
                  "codes_used": int(positive.sum()),
                  "code_perplexity": float(torch.exp(-(probability*probability.log()).sum())),
                  "per_image": records, "does_not_select_checkpoint": True}
        sheet.save(output_dir / f"{label}_input_reconstruction.png")
        (output_dir / f"{label}_metrics.json").write_text(
            json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
        print("Training-only stationary snapshot:", {key: value for key, value in report.items()
                                                     if key != "per_image"}, flush=True)
        return report
    finally:
        model.train(training)
        restore_rng(rng)

"""Bounded checkpoints through an ordinary mounted Drive folder.

Only this module's fixed slots and small pointers are overwritten. Legacy files
are read-only migration sources; no Google API, background upload, or deletion.
Use one writer per run. A mounted Drive write is not a server-side durability
guarantee, so keep the active Colab runtime until the save has finished syncing.
"""
import hashlib
import json
import math
import os
from pathlib import Path
import pickle
import re
import shutil
import time

import torch


def _digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def _copy(source, destination):
    # Overwrite a fixed inactive slot rather than creating/deleting generations.
    with Path(source).open("rb") as reader, Path(destination).open("wb") as writer:
        shutil.copyfileobj(reader, writer, length=8 * 1024**2)
        writer.flush()
        os.fsync(writer.fileno())


def _trusted_load(path):
    # Only use with this experiment's own checkpoints (pickle is not sandboxed).
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


class MountedCheckpointStore:
    """Two reusable resume slots, plus one logical best and its fallback slot."""

    def __init__(self, directory, local_cache, prefix):
        if prefix not in {"stage1", "stage2"}:
            raise ValueError("Checkpoint prefix must be stage1 or stage2")
        self.directory = Path(directory).resolve()
        self.local_cache = Path(local_cache).resolve() / prefix
        if self.directory == Path(self.directory.anchor):
            raise ValueError("Checkpoint directory must not be a filesystem root")
        if self.local_cache == self.directory or self.directory in self.local_cache.parents:
            raise ValueError("Local cache must be outside the mounted checkpoint directory")
        self.directory.mkdir(parents=True, exist_ok=True)
        self.local_cache.mkdir(parents=True, exist_ok=True)
        self.prefix, self.stage = prefix, int(prefix[-1])
        self.pointer = self.directory / f"{prefix}_mounted_latest.json"
        self.backup_pointer = self.directory / f"{prefix}_mounted_latest.previous.json"
        self._active = {"resume": None, "best": None}
        self._verified = {}
        self._best_checked, self._best_metadata = False, None
        self._declared_resume_updates = -1
        self._manifested_resume_files = set()

    def _path(self, name):
        if not isinstance(name, str) or not re.fullmatch(
                rf"{self.prefix}_(?:(?:resume|best)_[ab]|step\d{{9}}_[0-9a-f]{{8}})\.pth", name):
            raise ValueError("Checkpoint filename is outside this stage's exact scope")
        target = self.directory / name
        if target.resolve().parent != self.directory:
            raise ValueError("Checkpoint resolves outside its run directory")
        return target

    def _pointers(self, kind):
        if kind == "resume":
            return [self.pointer, self.backup_pointer,
                    self.directory / f"{self.prefix}_latest.json",
                    self.directory / f"{self.prefix}_latest.previous.json"]
        return [self.directory / f"{self.prefix}_best.json",
                self.directory / f"{self.prefix}_best.previous.json"]

    def _entries(self, kind="resume"):
        records = []
        seen = set()
        if kind == "resume":
            self._declared_resume_updates = -1
            self._manifested_resume_files = set()
        for pointer in self._pointers(kind):
            try:
                manifest = json.loads(pointer.read_text(encoding="utf-8"))
                entries = manifest["generations"]
                if not isinstance(entries, list):
                    continue
            except (OSError, ValueError, KeyError, TypeError):
                continue
            for entry in entries:
                try:
                    path = self._path(entry["file"])
                    if kind == "best" and not re.fullmatch(rf"{self.prefix}_best_[ab]\.pth", path.name):
                        continue
                    if kind == "resume" and "_best_" in path.name:
                        continue
                    if not isinstance(entry.get("sha256"), str) or not re.fullmatch("[0-9a-f]{64}", entry["sha256"]):
                        continue
                    if not isinstance(entry.get("bytes"), int) or entry["bytes"] <= 0:
                        continue
                    if "updates" in entry and (not isinstance(entry["updates"], int) or entry["updates"] < 0):
                        continue
                    if kind == "resume":
                        self._declared_resume_updates = max(self._declared_resume_updates, self._entry_step(entry))
                        self._manifested_resume_files.add(path.name)
                    if not path.is_file() or path.name in seen:
                        continue
                except (OSError, ValueError, KeyError, TypeError):
                    continue
                records.append(dict(entry))
                seen.add(path.name)
        if kind == "resume":
            # Unregistered legacy generations can survive a failed old manifest
            # update. Inspect them read-only, newest filename step first.
            for path in self.directory.glob(f"{self.prefix}_step*.pth"):
                if re.fullmatch(rf"{self.prefix}_step\d{{9}}_[0-9a-f]{{8}}\.pth", path.name) and path.name not in seen:
                    self._path(path.name)
                    records.append({"file": path.name, "bytes": path.stat().st_size})
        # Best pointer order is improvement order, not optimizer-step order:
        # after a rollback, a better score may legitimately have a lower step.
        return sorted(records, key=self._entry_step, reverse=True) if kind == "resume" else records

    @staticmethod
    def _entry_step(entry):
        match = re.search(r"_step(\d+)_", entry["file"])
        return int(entry.get("updates", match.group(1) if match else -1))

    def _validate(self, payload, kind, entry):
        if not isinstance(payload, dict) or payload.get("stage") != self.stage:
            raise ValueError("Checkpoint stage differs")
        if payload.get("engine_version") != "paper_faithful_v6":
            raise ValueError("Checkpoint engine differs")
        if not isinstance(payload.get("state_dict"), dict) or not payload["state_dict"]:
            raise ValueError("Checkpoint has no model state")
        progress = payload.get("progress")
        if not isinstance(progress, dict) or not isinstance(progress.get("updates"), int) or progress["updates"] < 0:
            raise ValueError("Checkpoint has no valid optimizer-step progress")
        if "updates" in entry and progress["updates"] != entry["updates"]:
            raise ValueError("Checkpoint step differs from pointer")
        filename_step = re.search(r"_step(\d+)_", entry["file"])
        if filename_step and int(filename_step.group(1)) != progress["updates"]:
            raise ValueError("Legacy checkpoint step differs from filename")
        if kind == "resume":
            for key in ("optimizer", "configuration", "rng"):
                if not isinstance(payload.get(key), dict):
                    raise ValueError(f"Resumable checkpoint lacks {key}")
            for key in ("epoch", "cursor"):
                if not isinstance(progress.get(key), int) or progress[key] < 0:
                    raise ValueError(f"Invalid resume {key}")
        else:
            selection = payload.get("selection", {})
            if (not isinstance(selection, dict) or selection.get("mode") != "min"
                    or not isinstance(selection.get("metric"), str)
                    or not math.isfinite(float(selection.get("value", math.nan)))):
                raise ValueError("Invalid best-checkpoint selection")

    def _read(self, entry, kind):
        source = self._path(entry["file"])
        digest = entry.get("sha256")
        if digest is None:
            cached = [key for key in self._verified
                      if key[0] == entry["file"] and key[2] == entry["bytes"]]
            if cached:
                digest = cached[0][1]
                entry["sha256"] = digest
        # Immutable local copies are reused during this process. Slots themselves
        # are never treated as immutable or trusted from a JSON pointer alone.
        key = (entry["file"], digest, entry["bytes"])
        local = self._verified.get(key)
        if local is None or not local.is_file():
            if source.stat().st_size != entry["bytes"]:
                raise ValueError("Checkpoint byte count differs")
            local = self.local_cache / f"{kind}_loaded.pth"
            self._verified = {key_: value for key_, value in self._verified.items() if value != local}
            _copy(source, local)
            actual = _digest(local)
            if local.stat().st_size != entry["bytes"] or (digest is not None and actual != digest):
                raise ValueError("Checkpoint SHA256 verification failed")
            # Only one verified loaded copy per kind is retained; stale cache
            # entries must not point at that reused path.
            entry["sha256"] = actual
            key = (entry["file"], actual, entry["bytes"])
            self._verified[key] = local
        payload = _trusted_load(local)
        self._validate(payload, kind, entry)
        entry["updates"] = payload["progress"]["updates"]
        self._active[kind] = dict(entry)
        return payload

    def _load_kind(self, kind):
        failures = []
        entries = self._entries(kind)
        verified_manifested = False
        failed_manifested = set()
        for entry in entries:
            if (kind == "resume" and entry["file"] not in self._manifested_resume_files
                    and self._entry_step(entry) < self._declared_resume_updates
                    and not verified_manifested):
                # A fresh mount may expose an old restored anchor before the
                # files promised by the actual run manifest become visible.
                # Never silently roll back to that unregistered anchor.
                for registered in entries:
                    if registered["file"] not in self._manifested_resume_files or registered["file"] in failed_manifested:
                        continue
                    try:
                        checked = self._read(registered, kind)
                        del checked
                        verified_manifested = True
                        break
                    except (OSError, ValueError, RuntimeError, EOFError, KeyError, TypeError, pickle.UnpicklingError):
                        failed_manifested.add(registered["file"])
                if not verified_manifested:
                    raise RuntimeError("Drive mount not visible or incomplete: newer manifested resume files "
                                       "cannot be verified; refusing to roll back to an unregistered old checkpoint. "
                                       "Wait for Drive to sync/remount; do not restart training from scratch.")
            try:
                payload = self._read(entry, kind)
                if kind == "resume" and entry["updates"] < self._declared_resume_updates:
                    print(f"Warning: newest declared update {self._declared_resume_updates} is unavailable/corrupt; "
                          f"using verified update {entry['updates']}.", flush=True)
                label = "Resuming committed checkpoint" if kind == "resume" else "Loaded best model"
                print(f"[checkpoint:{self.prefix}] {label}: {entry['file']} (update {entry['updates']})", flush=True)
                return payload
            except (OSError, ValueError, RuntimeError, EOFError, KeyError, TypeError, pickle.UnpicklingError) as error:
                failures.append(f"{entry['file']}: {error}")
                if kind == "resume" and entry["file"] in self._manifested_resume_files:
                    failed_manifested.add(entry["file"])
        known = failures or any(path.exists() for path in self._pointers(kind))
        if known:
            if kind == "resume" and self._declared_resume_updates >= 0 and not entries:
                raise RuntimeError("Drive mount not visible or incomplete: manifest exists but none of its resume "
                                   "files are visible. Wait for Drive sync/remount; refusing a silent restart.")
            raise RuntimeError(f"No recoverable {kind} checkpoint; refusing a silent restart. " + "; ".join(failures))
        return None

    def load(self):
        return self._load_kind("resume")

    def best_info(self):
        if not self._best_checked:
            payload = self._load_kind("best")
            if payload is not None:
                self._best_metadata = dict(payload["selection"])
            self._best_checked = True
        return None if self._best_metadata is None else dict(self._best_metadata)

    def load_best(self):
        return self._load_kind("best")

    def _write_pointer(self, kind, entries):
        primary, previous = self._pointers(kind)[:2]
        old = None
        try:
            old = primary.read_bytes()
            json.loads(old)
        except (OSError, ValueError):
            old = None
        raw = json.dumps({"format": "mounted_fixed_slots_v1", "generations": entries}, indent=2).encode("utf-8")
        # Backup is written before the primary; fixed tiny files avoid Trash churn.
        for path, content in ((previous, old), (primary, raw)):
            if content is not None:
                with path.open("wb") as handle:
                    handle.write(content)
                    handle.flush()
                    os.fsync(handle.fileno())

    def _save(self, payload, kind):
        self._validate(payload, kind, {"file": f"{self.prefix}_{kind}_a.pth"})
        if self._active[kind] is None and self._entries(kind):
            self._load_kind(kind)
        previous = self._active[kind]
        slot = "b" if previous and previous["file"] == f"{self.prefix}_{kind}_a.pth" else "a"
        name = f"{self.prefix}_{kind}_{slot}.pth"
        destination = self._path(name)
        local = self.local_cache / f"{kind}_pending.pth"
        started = time.monotonic()
        torch.save(payload, local)
        entry = {"file": name, "bytes": local.stat().st_size, "sha256": _digest(local),
                 "updates": payload["progress"]["updates"]}
        print(f"[checkpoint:{self.prefix}] writing {kind} update {entry['updates']}; {entry['bytes']/1024**3:.2f} GiB", flush=True)
        _copy(local, destination)
        if destination.stat().st_size != entry["bytes"]:
            raise RuntimeError(f"Checkpoint copy incomplete; local snapshot preserved at {local}")
        self._write_pointer(kind, [entry] + ([previous] if previous else []))
        self._active[kind] = dict(entry)
        # Cache only a path that will not be overwritten by the next save.
        self._verified = {key: value for key, value in self._verified.items() if value != local}
        label = "Checkpoint committed" if kind == "resume" else "Best model committed"
        print(f"[checkpoint:{self.prefix}] {label}: {name}; {time.monotonic()-started:.1f}s", flush=True)
        return entry

    def save(self, payload):
        return self._save(payload, "resume")

    def save_best(self, model_state_dict, progress, score, metric, metadata=None):
        score = float(score)
        if not math.isfinite(score) or not isinstance(metric, str) or not metric:
            raise ValueError("Best score must be finite and metric must be named")
        previous = self.best_info()
        if previous is not None:
            if previous["metric"] != metric:
                raise ValueError("Best-checkpoint metric changed within this run")
            if score >= previous["value"]:
                return None
        selection = {"metric": metric, "value": score, "mode": "min",
                     "source": "completed_epoch_training_mean", "epoch": int(progress["epoch"]),
                     "updates": int(progress["updates"])}
        payload = dict(metadata or {})
        payload.update({"engine_version": "paper_faithful_v6", "stage": self.stage,
                        "state_dict": model_state_dict, "selection": selection,
                        "progress": {"epoch": int(progress["epoch"]), "updates": int(progress["updates"]),
                                     "history": progress.get("history", [])}})
        entry = self._save(payload, "best")
        self._best_checked, self._best_metadata = True, dict(selection)
        print(f"[checkpoint:{self.prefix}] Best model: training {metric} {score:.6f}; epoch {selection['epoch']}; update {selection['updates']}", flush=True)
        return {**entry, **selection}


def export_final_model(model, destination, metadata, progress, local_cache):
    # Once per completed stage. Do not overwrite an already-complete parent on Run-all.
    destination = Path(destination)
    local = Path(local_cache) / destination.name
    local.parent.mkdir(parents=True, exist_ok=True)
    destination.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model_version": metadata["model_version"], "state_dict": model.state_dict(),
                "config": metadata, "epochs_completed": progress["epoch"],
                "training_history": progress["history"], "rng": capture_rng()}, local)
    partial = destination.with_suffix(destination.suffix + ".partial")
    with local.open("rb") as reader, partial.open("wb") as writer:
        shutil.copyfileobj(reader, writer, length=8 * 1024**2)
        writer.flush()
        os.fsync(writer.fileno())
    if partial.stat().st_size != local.stat().st_size:
        raise RuntimeError("Final model copy incomplete; local snapshot retained")
    partial.replace(destination)
    print("Final model saved:", destination, flush=True)

STAGE1_EPOCHS = 100  # Inferred initial budget; may be extended with the same run.
STAGE2_EPOCHS = 100
FAITHFUL_TRAINING_CONFIG = {
    "model_version": MODEL_VERSION,
    "micro_batch_size": 4,
    "effective_batch_size": 8,
    "num_workers": min(2, os.cpu_count() or 1),
    "gradient_clip_value": 1.0,
    "precision": "fp32",
    "stage2_reference_ber": (0.0, 1e-4, 1e-3, 5e-3, 1e-2),
    "codebook_size": 16384,
    "log_every_updates": 25,
    "checkpoint_every_updates": 1000,
    "dataset": DATA_MANIFEST,
    "semantic_model": SEMANTIC_MODEL_CONFIG,
    "activation_checkpointing": False,
}
FINAL_METADATA = {
    "model_version": MODEL_VERSION,
    "implementation_choices": {"input_size": 256, "K": 16384, "bits_per_index": 14},
    "reported": {"lr": 1e-4, "adam_betas": (.5, .9), "embed_dim": 4,
                 "loss": "MSE + .1 VGG-LPIPS + equal-weight two VQ terms", "train_counts": PAPER_TRAIN_COUNTS},
    "inferred_and_operational": FAITHFUL_TRAINING_CONFIG,
    "runtime": RUNTIME_INFO,
    "physical_evaluation_model": {"channel_version": PAPER_FSO_CHANNEL_VERSION,
                                  "config": asdict(PaperFSOSystemConfig()),
                                  "receiver_assumptions": PAPER_UNDISCLOSED_RECEIVER_ASSUMPTIONS,
                                  "dsp_version": SIGNAL_DSP_VERSION,
                                  "dsp_config": asdict(SignalDSPConfig())},
    "stage2_optimizer": "fresh Adam on entering Stage2; full state retained within each stage",
    "not_author_code": True,
}
CHECKPOINT_CACHE = LOCAL_CACHE / "checkpoint_staging"
print("Training configuration:", FAITHFUL_TRAINING_CONFIG)
print("Checkpoint files are several GB; reserve about 20 GB extra on Drive for two stages and temporary copies.")


def configure_activation_checkpointing(model, enabled=True):
    from torch.utils.checkpoint import checkpoint
    for module in model.modules():
        if isinstance(module, (ResidualBlock, AttnBlock, SpatialResBlock, SpatialAttnBlock)):
            original = getattr(module, "_uncheckpointed_forward", module.forward)
            module._uncheckpointed_forward = original
            def forward(*inputs, _original=original, _module=module):
                if enabled and _module.training and torch.is_grad_enabled():
                    return checkpoint(_original, *inputs, use_reentrant=False, preserve_rng_state=False)
                return _original(*inputs)
            module.forward = forward
    return model


def load_compatible_final(path, expected_configuration=None, expected_stage=None):
    payload = trusted_torch_load(path)
    expected_configuration = (FAITHFUL_TRAINING_CONFIG if expected_configuration is None
                              else expected_configuration)
    if (payload.get("model_version") != MODEL_VERSION
            or learning_configuration(payload.get("config", {}).get("inferred_and_operational", {})) != learning_configuration(expected_configuration)
            or (expected_stage is not None and payload.get("config", {}).get("stage") != expected_stage)):
        raise RuntimeError("Final model config/data differ; choose a different RUN_NAME")
    return payload

DIAGNOSTIC_TRAIN_INDICES = [0, 1000, 2000, 4000, 6000, 8000, 10000, 10100, 10200, 10400, 10600, 10800]
FINAL_METADATA["execution_revision"] = EXECUTION_REVISION
FINAL_METADATA["batch_noise_sampling"] = "independent per image, matching legacy microbatch-1"
print("New output folder:", MODEL_OUTPUT_DIR)


### Stage 1 — clean reconstruction training

In [ ]:
import gc

device = torch.device("cuda")
stage1_filename = str(MODEL_OUTPUT_DIR / "stage1_paper_protocol_final.pth")
stage1_store = MountedCheckpointStore(MODEL_OUTPUT_DIR / "resume", CHECKPOINT_CACHE, "stage1")
stage1_payload = (load_compatible_final(stage1_filename, expected_stage=1)
                  if Path(stage1_filename).exists() else None)
stage1_complete = stage1_payload is not None and stage1_payload["epochs_completed"] >= STAGE1_EPOCHS
if stage1_complete:
    model = configure_activation_checkpointing(FSO_VQVAE().to(device), enabled=False)
    model.load_state_dict(stage1_payload["state_dict"], strict=True)
    restore_rng(stage1_payload["rng"])
    print("Stage1 already complete; loaded final model without retraining.")
else:
    inspected = stage1_store.load()
    stage1_requires_resume = inspected is not None
    if inspected is None:
        if START_MODE == "resume" or stage_has_artifacts(MODEL_OUTPUT_DIR, 1):
            raise RuntimeError("No valid full Stage1 checkpoint. Refusing to restart from final weights or reset the run.")
        seed_fresh_training(FRESH_RUN_SEED)  # Only a genuinely new run, before model initialization.
    else:
        require_learning_compatibility(inspected["configuration"], FAITHFUL_TRAINING_CONFIG)
        print("Continue current run: update", inspected["progress"]["updates"],
              "completed epochs", inspected["progress"]["epoch"],
              "next image position", inspected["progress"]["cursor"], flush=True)
    model = configure_activation_checkpointing(FSO_VQVAE().to(device), enabled=False)
    if inspected is not None:
        model.load_state_dict(inspected["state_dict"], strict=True)
        restore_rng(inspected["rng"])
    del inspected
    gc.collect()
    chosen_micro, _ = choose_microbatch_no_update(model, combined_train, vgg_loss, device, requested=4)
    FAITHFUL_TRAINING_CONFIG["micro_batch_size"] = chosen_micro
    print(f"microbatch {chosen_micro}; effective batch8; FP32; save every1000 updates + epoch end", flush=True)
    progress1 = train_paper_stage(model, combined_train, vgg_loss, device, stage=1,
                                 epochs=STAGE1_EPOCHS, configuration=FAITHFUL_TRAINING_CONFIG,
                                 store=stage1_store, require_resume=stage1_requires_resume, best_metric="lpips")
    FINAL_METADATA["inferred_and_operational"] = FAITHFUL_TRAINING_CONFIG
    export_final_model(model, stage1_filename, {**FINAL_METADATA, "stage": 1}, progress1, CHECKPOINT_CACHE)
stage1_training_completed = True
del stage1_payload
gc.collect()
torch.cuda.empty_cache()


### Stage 2 — controlled quantized-error noise training

In [ ]:
import gc

device = torch.device("cuda")
stage1_filename = str(MODEL_OUTPUT_DIR / "stage1_paper_protocol_final.pth")
stage2_filename = str(MODEL_OUTPUT_DIR / "stage2_noise_training_final.pth")
if not Path(stage1_filename).exists():
    raise RuntimeError("Complete Stage1 before entering Stage2.")
parent_payload = load_compatible_final(stage1_filename, expected_stage=1)
if parent_payload["epochs_completed"] < STAGE1_EPOCHS:
    raise RuntimeError("Stage1 final is incomplete; resume Stage1 before entering Stage2.")
stage2_store = MountedCheckpointStore(MODEL_OUTPUT_DIR / "resume", CHECKPOINT_CACHE, "stage2")
stage2_configuration = {**FAITHFUL_TRAINING_CONFIG, "stage1_parent_sha256": sha256_path(stage1_filename)}
stage2_metadata = {**FINAL_METADATA, "stage": 2, "inferred_and_operational": stage2_configuration}
stage2_payload = (load_compatible_final(stage2_filename, stage2_configuration, expected_stage=2)
                  if Path(stage2_filename).exists() else None)
if not globals().get("stage1_training_completed", False):
    model = configure_activation_checkpointing(FSO_VQVAE().to(device), enabled=False)
if stage2_payload is not None and stage2_payload["epochs_completed"] >= STAGE2_EPOCHS:
    model.load_state_dict(stage2_payload["state_dict"], strict=True)
    restore_rng(stage2_payload["rng"])
    del parent_payload
    print("Stage2 already complete; loaded final model without retraining.")
else:
    inspected = stage2_store.load()
    stage2_requires_resume = inspected is not None
    if inspected is None:
        if stage_has_artifacts(MODEL_OUTPUT_DIR, 2):
            raise RuntimeError("Stage2 files exist but no valid full checkpoint is available; refusing a silent restart.")
        model.load_state_dict(parent_payload["state_dict"], strict=True)
        restore_rng(parent_payload["rng"])
        print("Starting Stage2 from the completed Stage1 final with a fresh Adam optimizer.")
    else:
        require_learning_compatibility(inspected["configuration"], stage2_configuration)
        model.load_state_dict(inspected["state_dict"], strict=True)
        restore_rng(inspected["rng"])
    del inspected, parent_payload
    gc.collect()
    chosen_micro, _ = choose_microbatch_no_update(model, combined_train, vgg_loss, device, requested=4)
    FAITHFUL_TRAINING_CONFIG["micro_batch_size"] = chosen_micro
    stage2_configuration["micro_batch_size"] = chosen_micro
    progress2 = train_paper_stage(model, combined_train, vgg_loss, device, stage=2,
                                 epochs=STAGE2_EPOCHS, configuration=stage2_configuration,
                                 store=stage2_store, require_resume=stage2_requires_resume, best_metric="lpips")
    export_final_model(model, stage2_filename, stage2_metadata, progress2, CHECKPOINT_CACHE)
stage2_training_completed = True
del stage2_payload
gc.collect()
torch.cuda.empty_cache()


## 5. 선택: 별도 1,000장 LPIPS 평가

학습·checkpoint 선택에 사용하지 않습니다. 아래 평가의 3회 반복·Alex backbone·256 crop은 본 재현 선택입니다. 수신 DSP가 실패한 프레임을 제외하거나 LPIPS를 임의의 1로 대체하지 않습니다.


In [ ]:
from torch.utils.data import DataLoader


def make_paper_evaluation_loader(image_dir, batch_size=1):
    dataset = VisionDataset(image_dir, paper_transform)
    if not len(dataset):
        raise RuntimeError("No evaluation images")
    return DataLoader(dataset, batch_size=batch_size, shuffle=False,
                      num_workers=min(2, os.cpu_count() or 1), pin_memory=True,
                      generator=torch.Generator().manual_seed(0))


def strict_json_value(value):
    if isinstance(value, dict):
        return {str(k): strict_json_value(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [strict_json_value(v) for v in value]
    if isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


@torch.inference_mode()
def evaluate_paper_lpips(model, loader, physical_index_channel, conditions,
                         channel_repeats=3, eval_backbone="alex"):
    if channel_repeats < 1:
        raise ValueError("channel_repeats must be positive")
    model.eval()
    metric = lpips.LPIPS(net=eval_backbone, version="0.1").to(device).eval()
    metric.requires_grad_(False)
    results = {}
    for condition in conditions:
        score_sum = 0.0
        realizations = 0
        totals = {"bit_errors": 0, "valid_bits": 0, "index_errors": 0,
                  "indices": 0, "symbol_errors": 0, "valid_symbols": 0}
        clipped_values = 0
        reconstruction_values = 0
        first_channel_diagnostics = None
        cfr_weak_frames = 0
        for images in loader:
            images = images.to(device, non_blocking=True)
            _, _, indices = model.encode(images)
            for _ in range(channel_repeats):
                received, statistics = physical_index_channel(
                    indices, return_statistics=True, **condition["channel_kwargs"])
                reconstruction = model.decode_indices(received, received.shape)
                if not bool(torch.isfinite(reconstruction).all()):
                    raise FloatingPointError("Nonfinite reconstruction; frames are not silently discarded")
                clipped_values += int((reconstruction.abs() > 1).sum())
                reconstruction_values += reconstruction.numel()
                score = metric(reconstruction.clamp(-1, 1), images).reshape(-1)
                score_sum += float(score.sum())
                realizations += score.numel()
                for key in totals:
                    totals[key] += statistics[key]
                channel = statistics.get("channel", {})
                if first_channel_diagnostics is None:
                    first_channel_diagnostics = channel
                strength = channel.get("signal_dsp", {}).get("cfr_peak_to_mean_power", [])
                cfr_weak_frames += sum(value < 10 for value in strength)
        results[condition["label"]] = {
            "lpips_mean": score_sum / realizations,
            "lpips_backbone": eval_backbone,
            "unique_images": len(loader.dataset),
            "channel_repeats": channel_repeats,
            "image_channel_realizations": realizations,
            "payload_ber": totals["bit_errors"] / max(totals["valid_bits"], 1),
            "index_error_rate": totals["index_errors"] / max(totals["indices"], 1),
            "symbol_error_rate": totals["symbol_errors"] / max(totals["valid_symbols"], 1),
            "reconstruction_clipped_fraction": clipped_values / max(reconstruction_values, 1),
            "weak_CFR_peak_frames_diagnostic_only": cfr_weak_frames,
            "first_realization_channel_diagnostics": first_channel_diagnostics,
            "condition": condition,
            "failure_policy": "all finite received frames retained, including failed carrier recovery",
        }
        print(condition["label"], results[condition["label"]]["lpips_mean"], flush=True)
    return results


paper_fso_receiver = PaperMalagaFSOChannel(PaperFSOSystemConfig())
paper_physical_index_channel = SignalDSPPaperIndexChannel(paper_fso_receiver)
# This separate comparator is deliberately labeled oracle, never used by default.
paper_oracle_index_channel = PaperFSOIndexChannel(paper_fso_receiver)

RUN_EVALUATION = False
if RUN_EVALUATION:
    if not globals().get("stage2_training_completed", False):
        raise RuntimeError("Load/complete the final Stage2 model before evaluation")
    loader = make_paper_evaluation_loader(GEE_TEST_DIR)
    if len(loader.dataset) != 1000:
        raise RuntimeError("Expected the separately prepared 1,000-image test set")
    results = evaluate_paper_lpips(model, loader, paper_physical_index_channel,
                                   paper_fig4_conditions(), channel_repeats=3,
                                   eval_backbone="alex")
    report = {"results": results, "runtime": RUNTIME_INFO,
              "evaluation_set": "separate NAIP proxy, not author-released images",
              "receiver": paper_physical_index_channel.reproducibility_config()}
    target = MODEL_OUTPUT_DIR / "evaluation_fig4_alex.json"
    target.write_text(json.dumps(strict_json_value(report), ensure_ascii=False,
                                 indent=2, allow_nan=False), encoding="utf-8")
    print("Evaluation saved:", target)
else:
    print("Evaluation is not run automatically. Set RUN_EVALUATION=True after training.")
    print("Other sweeps: paper_fig6_pointing_conditions(), paper_fig6_background_conditions(),")
    print("paper_fig7_weather_conditions(), paper_fig8_turbulence_conditions().")
    print("Signal-estimated receiver supports single Rx only. Diversity helpers are oracle baselines, not identical DSP.")


In [ ]:
# Optional Colab cleanup. Keep disabled so an interrupted training cell cannot be
# followed by a misleading "all work completed" message or automatic disconnect.
# from google.colab import runtime
# runtime.unassign()